# Camada Gold — indicadores e cenários de reposição de renda

## 1. Objetivo

Este notebook constrói a camada Gold do projeto sobre poupança previdenciária complementar no Brasil, com enfoque na Região Sudeste.

A camada Gold utiliza os dados previamente tratados nas camadas Bronze e Silver para produzir indicadores analíticos, cruzamentos socioeconômicos e cenários de reposição da renda.

Serão considerados:

- previdência complementar aberta, supervisionada pela SUSEP;
- previdência complementar fechada, supervisionada pela PREVIC;
- população e rendimento médio mensal dos estados da Região Sudeste;
- indicadores de participantes, beneficiários, contribuições, resgates e patrimônio;
- cenários ilustrativos de acumulação e reposição da renda.

As simulações apresentadas representam cenários baseados em premissas documentadas e não constituem garantia de rentabilidade ou benefício futuro.


In [0]:

# Importa as funções utilizadas para agregações,
# transformações, cálculos e validações.
from pyspark.sql import functions as F
from pyspark.sql.types import DecimalType
from pyspark.sql.window import Window

# Define um tipo decimal padronizado para cálculos monetários.
tipo_monetario = DecimalType(20, 2)

print("Ambiente da camada Gold preparado com sucesso.")

Ambiente da camada Gold preparado com sucesso.


In [0]:
# Carrega os dados da previdência aberta por unidade da Federação.
df_susep_uf = spark.table(
    "workspace.silver.susep_previdencia_uf"
)

# Carrega os dados de rendimento médio mensal do IBGE.
df_ibge_rendimento = spark.table(
    "workspace.silver.ibge_rendimento_7444_sudeste"
)

# Carrega os dados populacionais do IBGE.
df_ibge_populacao = spark.table(
    "workspace.silver.ibge_populacao_6407_sudeste"
)

# Carrega as informações populacionais das entidades
# de previdência complementar fechada.
df_previc_dsi = spark.table(
    "workspace.silver.previc_dsi_2025"
)

# Carrega as informações dos planos e entidades
# de previdência complementar fechada.
df_previc_epb = spark.table(
    "workspace.silver.previc_epb_2025"
)

print("Tabelas Silver carregadas com sucesso.")

Tabelas Silver carregadas com sucesso.


## 2. Inspeção das fontes da camada Gold

Antes da construção dos indicadores, são verificadas as estruturas das principais tabelas Silver. Essa inspeção permite identificar as dimensões e métricas disponíveis para cada segmento previdenciário.

In [0]:
# Exibe a estrutura das principais tabelas que serão
# utilizadas na construção dos indicadores Gold.

print("SUSEP — previdência por UF:")
df_susep_uf.printSchema()

print("\nIBGE — rendimento:")
df_ibge_rendimento.printSchema()

print("\nIBGE — população:")
df_ibge_populacao.printSchema()

print("\nPREVIC — informações populacionais:")
df_previc_dsi.printSchema()

print("\nPREVIC — planos e entidades:")
df_previc_epb.printSchema()

SUSEP — previdência por UF:
root
 |-- codigo_fip: string (nullable = true)
 |-- ano_mes: string (nullable = true)
 |-- uf: string (nullable = true)
 |-- contribuicao_informada: string (nullable = true)
 |-- beneficio_pago_informado: string (nullable = true)
 |-- resgate_pago_informado: string (nullable = true)
 |-- participantes_informado: string (nullable = true)
 |-- beneficiarios_informado: string (nullable = true)
 |-- resgates_informado: string (nullable = true)
 |-- tipo_produto: string (nullable = true)
 |-- _arquivo_origem: string (nullable = true)
 |-- _data_ingestao: timestamp (nullable = true)
 |-- data_referencia: date (nullable = true)
 |-- contribuicao: decimal(20,2) (nullable = true)
 |-- beneficio_pago: decimal(20,2) (nullable = true)
 |-- resgate_pago: decimal(20,2) (nullable = true)
 |-- participantes_original: long (nullable = true)
 |-- beneficiarios_original: long (nullable = true)
 |-- resgates_original: long (nullable = true)
 |-- participantes: long (nullable = 

## 3. Contexto socioeconômico da Região Sudeste

A primeira tabela Gold reúne a população residente e o rendimento médio mensal das quatro unidades da Federação da Região Sudeste.

Para evitar dupla contagem, são selecionados apenas:

- a categoria de sexo `TOTAL` nas duas fontes;
- o grupo de idade `Total` na tabela populacional;
- os anos comuns às duas bases.

O cruzamento é realizado por código da unidade da Federação e ano, produzindo uma linha para cada combinação de estado e período.

In [0]:
# Mapeia os códigos oficiais do IBGE para as siglas
# das unidades da Federação da Região Sudeste.
mapa_siglas_uf = F.create_map(
    F.lit("31"), F.lit("MG"),
    F.lit("32"), F.lit("ES"),
    F.lit("33"), F.lit("RJ"),
    F.lit("35"), F.lit("SP")
)

# Seleciona o rendimento médio mensal total,
# evitando separar os dados por homens e mulheres.
df_gold_rendimento_uf_ano = (
    df_ibge_rendimento
    .filter(F.upper(F.trim(F.col("sexo"))) == "TOTAL")
    .select(
        "codigo_uf",
        "uf",
        "ano",
        F.col("rendimento_medio_mensal").alias(
            "rendimento_medio_mensal_reais"
        )
    )
)

# Seleciona a população total de cada estado e ano.
# O filtro por sexo TOTAL e grupo de idade Total impede
# a soma repetida de categorias populacionais.
df_gold_populacao_uf_ano = (
    df_ibge_populacao
    .filter(
        (F.upper(F.trim(F.col("sexo"))) == "TOTAL")
        & (F.upper(F.trim(F.col("grupo_idade"))) == "TOTAL")
    )
    .select(
        "codigo_uf",
        "ano",
        F.col("populacao_pessoas").alias("populacao_total")
    )
)

# Combina rendimento e população utilizando o código
# da unidade da Federação e o ano como chaves.
df_gold_contexto_socioeconomico = (
    df_gold_rendimento_uf_ano.alias("renda")
    .join(
        df_gold_populacao_uf_ano.alias("populacao"),
        on=["codigo_uf", "ano"],
        how="inner"
    )
    .select(
        F.col("codigo_uf"),
        mapa_siglas_uf[F.col("codigo_uf")].alias("sigla_uf"),
        F.col("renda.uf").alias("uf"),
        F.lit("Sudeste").alias("regiao"),
        F.col("ano"),
        F.col("populacao.populacao_total"),
        F.col("renda.rendimento_medio_mensal_reais")
    )

     # Acrescenta a data de processamento para rastreabilidade.
    .withColumn("data_processamento", F.current_timestamp())
    .orderBy("ano", "sigla_uf")
)

display(df_gold_contexto_socioeconomico)

codigo_uf,sigla_uf,uf,regiao,ano,populacao_total,rendimento_medio_mensal_reais,data_processamento
32,ES,Espírito Santo,Sudeste,2012,3711000,2820.00,2026-09-13T00:22:57.393Z
31,MG,Minas Gerais,Sudeste,2012,20084000,2739.00,2026-09-13T00:22:57.393Z
33,RJ,Rio de Janeiro,Sudeste,2012,16719000,3332.00,2026-09-13T00:22:57.393Z
35,SP,São Paulo,Sudeste,2012,43157000,3994.00,2026-09-13T00:22:57.393Z
32,ES,Espírito Santo,Sudeste,2013,3750000,3024.00,2026-09-13T00:22:57.393Z
31,MG,Minas Gerais,Sudeste,2013,20204000,2816.00,2026-09-13T00:22:57.393Z
33,RJ,Rio de Janeiro,Sudeste,2013,16800000,3427.00,2026-09-13T00:22:57.393Z
35,SP,São Paulo,Sudeste,2013,43515000,4145.00,2026-09-13T00:22:57.393Z
32,ES,Espírito Santo,Sudeste,2014,3790000,3144.00,2026-09-13T00:22:57.393Z
31,MG,Minas Gerais,Sudeste,2014,20330000,2928.00,2026-09-13T00:22:57.393Z


### 3.1 Validação e gravação do contexto socioeconômico

A tabela é validada quanto à cobertura geográfica e temporal, unicidade das chaves e presença de valores populacionais e monetários válidos.

A população total e o rendimento médio são mantidos como indicadores independentes, pois possuem universos estatísticos distintos. Portanto, não são multiplicados entre si.

In [0]:
# Consolida as validações da primeira tabela Gold.
resultado_contexto = (
    df_gold_contexto_socioeconomico
    .agg(
        F.count("*").alias("registros"),
        F.countDistinct("codigo_uf").alias("ufs_distintas"),
        F.min("ano").alias("primeiro_ano"),
        F.max("ano").alias("ultimo_ano"),

        # Verifica ausência de dados necessários.
        F.sum(
            F.when(F.col("populacao_total").isNull(), 1).otherwise(0)
        ).alias("populacoes_nulas"),
        F.sum(
            F.when(
                F.col("rendimento_medio_mensal_reais").isNull(),
                1
            ).otherwise(0)
        ).alias("rendimentos_nulos"),

        # Verifica valores incompatíveis com os indicadores.
        F.sum(
            F.when(F.col("populacao_total") <= 0, 1).otherwise(0)
        ).alias("populacoes_nao_positivas"),
        F.sum(
            F.when(
                F.col("rendimento_medio_mensal_reais") <= 0,
                1
            ).otherwise(0)
        ).alias("rendimentos_nao_positivos")
    )
)

display(resultado_contexto)

registros,ufs_distintas,primeiro_ano,ultimo_ano,populacoes_nulas,rendimentos_nulos,populacoes_nao_positivas,rendimentos_nao_positivos
56,4,2012,2025,0,0,0,0


In [0]:
# Verifica se existe mais de uma linha para a mesma UF e ano.
df_repeticoes_contexto = (
    df_gold_contexto_socioeconomico
    .groupBy("codigo_uf", "ano")
    .count()
    .filter(F.col("count") > 1)
    .agg(
        F.count("*").alias("grupos_repetidos"),
        F.sum(F.col("count") - 1).alias("linhas_excedentes"),
        F.max("count").alias("maior_repeticao")
    )
)

display(df_repeticoes_contexto)

grupos_repetidos,linhas_excedentes,maior_repeticao
0,null,null


In [0]:
# Recupera os resultados calculados nas validações anteriores.
validacao_contexto = resultado_contexto.first()
validacao_repeticoes = df_repeticoes_contexto.first()

# Interrompe a execução caso algum critério obrigatório
# de qualidade não seja atendido.
assert validacao_contexto["registros"] == 56, \
    "Quantidade inesperada de registros."

assert validacao_contexto["ufs_distintas"] == 4, \
    "Quantidade inesperada de UFs."

assert validacao_contexto["primeiro_ano"] == 2012, \
    "Primeiro ano inesperado."

assert validacao_contexto["ultimo_ano"] == 2025, \
    "Último ano inesperado."

assert validacao_contexto["populacoes_nulas"] == 0, \
    "Foram encontradas populações nulas."

assert validacao_contexto["rendimentos_nulos"] == 0, \
    "Foram encontrados rendimentos nulos."

assert validacao_contexto["populacoes_nao_positivas"] == 0, \
    "Foram encontradas populações não positivas."

assert validacao_contexto["rendimentos_nao_positivos"] == 0, \
    "Foram encontrados rendimentos não positivos."

assert validacao_repeticoes["grupos_repetidos"] == 0, \
    "Foram encontradas chaves repetidas."

# Garante que o schema Gold exista no catálogo.
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.gold")

# Define o nome da tabela de destino.
tabela_gold_contexto = (
    "workspace.gold.contexto_socioeconomico_uf_ano"
)

# Grava a tabela em formato Delta.
# A gravação somente acontece após todas as validações anteriores.
(
    df_gold_contexto_socioeconomico
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(tabela_gold_contexto)
)

print(
    "Validações aprovadas e tabela Gold gravada com sucesso:",
    tabela_gold_contexto
)

Validações aprovadas e tabela Gold gravada com sucesso: workspace.gold.contexto_socioeconomico_uf_ano


In [0]:
# Lê novamente a tabela diretamente da camada Gold.
df_gold_contexto_gravado = spark.table(
    "workspace.gold.contexto_socioeconomico_uf_ano"
)

# Valida a tabela armazenada no catálogo.
df_validacao_contexto_gravado = (
    df_gold_contexto_gravado
    .agg(
        F.count("*").alias("quantidade_registros"),
        F.countDistinct("sigla_uf").alias("ufs_distintas"),
        F.min("ano").alias("primeiro_ano"),
        F.max("ano").alias("ultimo_ano")
    )
)

display(df_validacao_contexto_gravado)

quantidade_registros,ufs_distintas,primeiro_ano,ultimo_ano
56,4,2012,2025


## 4. Indicadores anuais da previdência aberta por UF

Os dados mensais da SUSEP são consolidados por unidade da Federação, produto e período.

As variáveis monetárias e a quantidade de resgates representam movimentações e serão posteriormente acumuladas ao longo do ano. Participantes e beneficiários representam estoques e, por isso, serão obtidos a partir do último mês disponível de cada ano, evitando a soma indevida da mesma população ao longo dos meses.

O recorte considera PGBL e previdência tradicional, que são os produtos disponíveis na fonte por unidade da Federação.

In [0]:
# Importa novamente as funções do PySpark,
# pois a sessão Python foi reiniciada.
from pyspark.sql import functions as F

# Recarrega a tabela Silver da SUSEP diretamente do catálogo.
# Isso permite executar esta etapa mesmo após o reinício da sessão.
df_susep_uf = spark.table(
    "workspace.silver.susep_previdencia_uf"
)

# Define as unidades da Federação pertencentes à Região Sudeste.
ufs_sudeste = ["ES", "MG", "RJ", "SP"]

# Prepara os dados mensais da SUSEP para a construção
# dos indicadores anuais da previdência aberta.
df_susep_uf_base_gold = (
    df_susep_uf

    # Mantém somente os quatro estados da Região Sudeste
    # e o período compatível com as tabelas do IBGE.
    .filter(
        F.col("uf").isin(ufs_sudeste)
        & (F.substring("ano_mes", 1, 4).cast("int") >= 2012)
        & (F.substring("ano_mes", 1, 4).cast("int") <= 2025)
    )

    # Extrai ano e mês do campo ano_mes, originalmente YYYYMM.
    .withColumn(
        "ano",
        F.substring("ano_mes", 1, 4).cast("int")
    )
    .withColumn(
        "mes",
        F.substring("ano_mes", 5, 2).cast("int")
    )

    # Padroniza as dimensões utilizadas nas agregações.
    .withColumn("uf", F.upper(F.trim(F.col("uf"))))
    .withColumn(
        "tipo_produto",
        F.upper(F.trim(F.col("tipo_produto")))
    )
)

# Consolida as empresas de um mesmo produto, UF e mês.
# Valores monetários e quantidades de movimentações são somados.
# Participantes e beneficiários também são somados dentro do mês,
# pois cada linha representa uma entidade/código FIP.
df_susep_uf_mensal_gold = (
    df_susep_uf_base_gold
    .groupBy(
        "ano_mes",
        "ano",
        "mes",
        "uf",
        "tipo_produto"
    )
    .agg(
        F.sum("contribuicao").alias("contribuicao_mensal"),
        F.sum("beneficio_pago").alias("beneficio_pago_mensal"),
        F.sum("resgate_pago").alias("resgate_pago_mensal"),
        F.sum("participantes").alias("participantes_mes"),
        F.sum("beneficiarios").alias("beneficiarios_mes"),
        F.sum("quantidade_resgates").alias(
            "quantidade_resgates_mensal"
        ),
        F.countDistinct("codigo_fip").alias(
            "entidades_informantes"
        )
    )
    .orderBy("ano", "mes", "uf", "tipo_produto")
)

display(df_susep_uf_mensal_gold)

ano_mes,ano,mes,uf,tipo_produto,contribuicao_mensal,beneficio_pago_mensal,resgate_pago_mensal,participantes_mes,beneficiarios_mes,quantidade_resgates_mensal,entidades_informantes
201201,2012,1,ES,PGBL,7646673.34,null,4323507.45,33771,null,646,17
201201,2012,1,ES,PREVTRAD,3704095.38,760096.01,767034.48,67473,923,150,36
201201,2012,1,MG,PGBL,36992336.26,null,16596973.14,104678,null,1913,18
201201,2012,1,MG,PREVTRAD,25924806.18,6617598.08,5253588.86,517821,4960,396,40
201201,2012,1,RJ,PGBL,71237992.06,null,34987802.36,158756,null,3409,20
201201,2012,1,RJ,PREVTRAD,40163998.20,13065545.08,7756279.23,631680,17041,795,45
201201,2012,1,SP,PGBL,291343281.02,null,431205247.79,709517,null,14884,21
201201,2012,1,SP,PREVTRAD,108419716.90,51449007.79,72477797.67,1276283,23895,3203,41
201202,2012,2,ES,PGBL,7092811.96,null,4489635.42,31915,null,466,15
201202,2012,2,ES,PREVTRAD,3483653.29,558492.78,628718.63,62853,923,104,38


### 4.1 Consolidação anual dos indicadores da SUSEP

As contribuições, benefícios pagos, resgates e quantidades de resgates são variáveis de fluxo. Portanto, seus valores mensais são somados para formar os indicadores anuais.

Participantes e beneficiários são variáveis de estoque. Para evitar sua soma ao longo dos meses, são utilizados os valores do último mês disponível em cada ano.

Valores nulos decorrentes da ausência estrutural de determinada informação para um produto são preservados e não interpretados automaticamente como zero.

In [0]:
# Importa a função de janela, caso a sessão tenha sido reiniciada.
from pyspark.sql.window import Window

# Consolida os fluxos mensais ao longo de cada ano.
df_susep_fluxos_anuais = (
    df_susep_uf_mensal_gold
    .groupBy("ano", "uf", "tipo_produto")
    .agg(
        # Soma as movimentações monetárias dos meses do ano.
        F.sum("contribuicao_mensal").alias(
            "contribuicao_anual"
        ),
        F.sum("beneficio_pago_mensal").alias(
            "beneficio_pago_anual"
        ),
        F.sum("resgate_pago_mensal").alias(
            "resgate_pago_anual"
        ),

        # Soma a quantidade de resgates ocorridos no ano.
        F.sum("quantidade_resgates_mensal").alias(
            "quantidade_resgates_anual"
        ),

        # Confere quantos meses estão presentes em cada grupo.
        F.countDistinct("mes").alias("meses_com_dados")
    )
)

# Define uma janela ordenada do mês mais recente
# para o mais antigo dentro de cada UF, produto e ano.
janela_ultimo_mes = (
    Window
    .partitionBy("ano", "uf", "tipo_produto")
    .orderBy(F.col("mes").desc())
)

# Seleciona o último mês disponível de cada ano.
# Participantes e beneficiários são estoques e não devem
# ser somados ao longo dos 12 meses.
df_susep_estoques_fim_ano = (
    df_susep_uf_mensal_gold
    .withColumn(
        "ordem_mes",
        F.row_number().over(janela_ultimo_mes)
    )
    .filter(F.col("ordem_mes") == 1)
    .select(
        "ano",
        "uf",
        "tipo_produto",
        F.col("ano_mes").alias("periodo_referencia_estoque"),
        F.col("mes").alias("ultimo_mes_disponivel"),
        F.col("participantes_mes").alias(
            "participantes_fim_ano"
        ),
        F.col("beneficiarios_mes").alias(
            "beneficiarios_fim_ano"
        ),
        F.col("entidades_informantes").alias(
            "entidades_informantes_ultimo_mes"
        )
    )
)

# Combina os fluxos anuais com os estoques do último mês.
df_gold_previdencia_aberta_uf_ano = (
    df_susep_fluxos_anuais.alias("fluxos")
    .join(
        df_susep_estoques_fim_ano.alias("estoques"),
        on=["ano", "uf", "tipo_produto"],
        how="inner"
    )
    .select(
        "ano",
        "uf",
        "tipo_produto",
        "meses_com_dados",
        "periodo_referencia_estoque",
        "ultimo_mes_disponivel",
        "contribuicao_anual",
        "beneficio_pago_anual",
        "resgate_pago_anual",
        "quantidade_resgates_anual",
        "participantes_fim_ano",
        "beneficiarios_fim_ano",
        "entidades_informantes_ultimo_mes"
    )

    # Acrescenta a região e a data de processamento.
    .withColumn("regiao", F.lit("Sudeste"))
    .withColumn("data_processamento", F.current_timestamp())

    # Organiza os indicadores por ano, UF e produto.
    .orderBy("ano", "uf", "tipo_produto")
)

display(df_gold_previdencia_aberta_uf_ano)

ano,uf,tipo_produto,meses_com_dados,periodo_referencia_estoque,ultimo_mes_disponivel,contribuicao_anual,beneficio_pago_anual,resgate_pago_anual,quantidade_resgates_anual,participantes_fim_ano,beneficiarios_fim_ano,entidades_informantes_ultimo_mes,regiao,data_processamento
2012,ES,PGBL,12,201212,12,96947396.16,null,48684991.86,5731,36177,null,18,Sudeste,2026-09-13T01:33:53.636Z
2012,ES,PREVTRAD,12,201212,12,44137529.45,12428095.72,11706045.58,1191,64023,869,37,Sudeste,2026-09-13T01:33:53.636Z
2012,MG,PGBL,12,201212,12,484803388.84,null,211647011.58,20729,106243,null,19,Sudeste,2026-09-13T01:33:53.636Z
2012,MG,PREVTRAD,12,201212,12,287767270.24,91861237.08,56461182.18,4120,404438,4677,40,Sudeste,2026-09-13T01:33:53.636Z
2012,RJ,PGBL,12,201212,12,961132114.63,null,447780125.00,35240,162744,null,20,Sudeste,2026-09-13T01:33:53.636Z
2012,RJ,PREVTRAD,12,201212,12,451733055.78,176869977.53,106221327.08,62685,533359,16330,47,Sudeste,2026-09-13T01:33:53.636Z
2012,SP,PGBL,12,201212,12,3924069664.57,null,2091783647.07,144362,939006,null,20,Sudeste,2026-09-13T01:33:53.636Z
2012,SP,PREVTRAD,12,201212,12,1301963816.57,642888964.72,820786261.70,326554,1371059,23458,43,Sudeste,2026-09-13T01:33:53.636Z
2013,ES,PGBL,12,201312,12,103710010.36,null,62624275.34,5739,41442,null,18,Sudeste,2026-09-13T01:33:53.636Z
2013,ES,PREVTRAD,12,201312,12,51196750.45,12135590.65,12072866.68,1085,74137,881,38,Sudeste,2026-09-13T01:33:53.636Z


### 4.2 Validação dos indicadores anuais da previdência aberta

A tabela anual é verificada quanto à cobertura temporal, geográfica e por produto, à presença dos 12 meses, à utilização de dezembro como referência dos estoques e à unicidade das chaves.

Os valores nulos de benefícios e beneficiários do PGBL são avaliados separadamente, pois representam ausência estrutural da informação na fonte e não falha de processamento.

In [0]:
# Valida a estrutura e a cobertura da tabela anual da SUSEP.
df_validacao_previdencia_aberta = (
    df_gold_previdencia_aberta_uf_ano
    .agg(
        F.count("*").alias("registros"),
        F.countDistinct("uf").alias("ufs_distintas"),
        F.countDistinct("tipo_produto").alias("produtos_distintos"),
        F.min("ano").alias("primeiro_ano"),
        F.max("ano").alias("ultimo_ano"),

        # Verifica se todos os grupos possuem 12 meses.
        F.min("meses_com_dados").alias("menor_quantidade_meses"),
        F.max("meses_com_dados").alias("maior_quantidade_meses"),

        # Verifica o mês utilizado como referência do estoque.
        F.min("ultimo_mes_disponivel").alias("menor_ultimo_mes"),
        F.max("ultimo_mes_disponivel").alias("maior_ultimo_mes"),

        # Confere os principais fluxos anuais.
        F.sum(
            F.when(F.col("contribuicao_anual").isNull(), 1).otherwise(0)
        ).alias("contribuicoes_nulas"),
        F.sum(
            F.when(F.col("resgate_pago_anual").isNull(), 1).otherwise(0)
        ).alias("resgates_nulos")
    )
)

display(df_validacao_previdencia_aberta)

registros,ufs_distintas,produtos_distintos,primeiro_ano,ultimo_ano,menor_quantidade_meses,maior_quantidade_meses,menor_ultimo_mes,maior_ultimo_mes,contribuicoes_nulas,resgates_nulos
112,4,2,2012,2025,12,12,12,12,0,0


In [0]:
# Separa os valores nulos por produto para distinguir
# ausência estrutural da informação de eventual falha.
df_validacao_produtos_aberta = (
    df_gold_previdencia_aberta_uf_ano
    .groupBy("tipo_produto")
    .agg(
        F.count("*").alias("registros"),
        F.sum(
            F.when(F.col("beneficio_pago_anual").isNull(), 1).otherwise(0)
        ).alias("beneficios_nulos"),
        F.sum(
            F.when(F.col("participantes_fim_ano").isNull(), 1).otherwise(0)
        ).alias("participantes_nulos"),
        F.sum(
            F.when(F.col("beneficiarios_fim_ano").isNull(), 1).otherwise(0)
        ).alias("beneficiarios_nulos"),
        F.sum(
            F.when(
                F.col("quantidade_resgates_anual").isNull(),
                1
            ).otherwise(0)
        ).alias("quantidades_resgates_nulas")
    )
    .orderBy("tipo_produto")
)

display(df_validacao_produtos_aberta)

tipo_produto,registros,beneficios_nulos,participantes_nulos,beneficiarios_nulos,quantidades_resgates_nulas
PGBL,56,56,0,56,0
PREVTRAD,56,0,0,0,0


In [0]:
# Verifica a unicidade da chave ano, UF e produto.
df_repeticoes_previdencia_aberta = (
    df_gold_previdencia_aberta_uf_ano
    .groupBy("ano", "uf", "tipo_produto")
    .count()
    .filter(F.col("count") > 1)
    .agg(
        F.count("*").alias("grupos_repetidos"),
        F.sum(F.col("count") - 1).alias("linhas_excedentes"),
        F.max("count").alias("maior_repeticao")
    )
)

display(df_repeticoes_previdencia_aberta)

grupos_repetidos,linhas_excedentes,maior_repeticao
0,null,null


In [0]:
# Recupera os resultados das validações anteriores.
validacao_aberta = df_validacao_previdencia_aberta.first()
validacao_repeticoes_aberta = (
    df_repeticoes_previdencia_aberta.first()
)

# Converte a validação por produto em um dicionário
# para verificar PGBL e PREVTRAD separadamente.
validacao_por_produto = {
    linha["tipo_produto"]: linha
    for linha in df_validacao_produtos_aberta.collect()
}

# Confirma a estrutura geral da tabela.
assert validacao_aberta["registros"] == 112, \
    "Quantidade inesperada de registros."

assert validacao_aberta["ufs_distintas"] == 4, \
    "Quantidade inesperada de UFs."

assert validacao_aberta["produtos_distintos"] == 2, \
    "Quantidade inesperada de produtos."

assert validacao_aberta["primeiro_ano"] == 2012, \
    "Primeiro ano inesperado."

assert validacao_aberta["ultimo_ano"] == 2025, \
    "Último ano inesperado."

assert validacao_aberta["menor_quantidade_meses"] == 12, \
    "Existe grupo com menos de 12 meses."

assert validacao_aberta["maior_quantidade_meses"] == 12, \
    "Existe grupo com quantidade inesperada de meses."

assert validacao_aberta["menor_ultimo_mes"] == 12, \
    "Existe estoque sem referência em dezembro."

assert validacao_aberta["maior_ultimo_mes"] == 12, \
    "Foi encontrado mês de referência inesperado."

assert validacao_aberta["contribuicoes_nulas"] == 0, \
    "Foram encontradas contribuições anuais nulas."

assert validacao_aberta["resgates_nulos"] == 0, \
    "Foram encontrados resgates anuais nulos."

assert validacao_repeticoes_aberta["grupos_repetidos"] == 0, \
    "Foram encontradas chaves repetidas."

# Confirma que os nulos do PGBL são apenas os esperados
# em benefícios e beneficiários.
assert validacao_por_produto["PGBL"]["beneficios_nulos"] == 56, \
    "Estrutura inesperada nos benefícios do PGBL."

assert validacao_por_produto["PGBL"]["beneficiarios_nulos"] == 56, \
    "Estrutura inesperada nos beneficiários do PGBL."

assert validacao_por_produto["PGBL"]["participantes_nulos"] == 0, \
    "Foram encontrados participantes nulos no PGBL."

# Confirma que o PREVTRAD possui os campos populacionais.
assert validacao_por_produto["PREVTRAD"]["beneficios_nulos"] == 0, \
    "Foram encontrados benefícios nulos no PREVTRAD."

assert validacao_por_produto["PREVTRAD"]["beneficiarios_nulos"] == 0, \
    "Foram encontrados beneficiários nulos no PREVTRAD."

assert validacao_por_produto["PREVTRAD"]["participantes_nulos"] == 0, \
    "Foram encontrados participantes nulos no PREVTRAD."

# Define o nome da segunda tabela Gold.
tabela_gold_previdencia_aberta = (
    "workspace.gold.previdencia_aberta_uf_ano"
)

# Grava os indicadores anuais em formato Delta.
(
    df_gold_previdencia_aberta_uf_ano
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(tabela_gold_previdencia_aberta)
)

print(
    "Validações aprovadas e tabela Gold gravada com sucesso:",
    tabela_gold_previdencia_aberta
)

Validações aprovadas e tabela Gold gravada com sucesso: workspace.gold.previdencia_aberta_uf_ano


In [0]:
# Lê novamente a tabela diretamente do catálogo Gold.
df_previdencia_aberta_gravado = spark.table(
    "workspace.gold.previdencia_aberta_uf_ano"
)

# Confere os principais indicadores da tabela armazenada.
df_validacao_aberta_gravado = (
    df_previdencia_aberta_gravado
    .agg(
        F.count("*").alias("quantidade_registros"),
        F.countDistinct("uf").alias("ufs_distintas"),
        F.countDistinct("tipo_produto").alias("produtos_distintos"),
        F.min("ano").alias("primeiro_ano"),
        F.max("ano").alias("ultimo_ano")
    )
)

display(df_validacao_aberta_gravado)

quantidade_registros,ufs_distintas,produtos_distintos,primeiro_ano,ultimo_ano
112,4,2,2012,2025


## 5. Indicadores da previdência complementar fechada

A Superintendência Nacional de Previdência Complementar — PREVIC — é responsável pela supervisão das Entidades Fechadas de Previdência Complementar (EFPC).

Nesta etapa são utilizadas informações provenientes do Sistema de Captação de Dados de População e Benefícios (EPB). Esse sistema é utilizado pelas entidades para informar à PREVIC dados e atualizações relacionados às populações e aos benefícios administrados pelos planos de previdência complementar fechada.

As tabelas analisadas contêm dois conjuntos principais de informações:

- dados populacionais, com participantes ativos, aposentados e beneficiários de pensão classificados por sexo e faixa etária;
- dados dos planos, com quantidades iniciais, entradas, saídas e quantidades finais relacionadas aos diferentes tipos de benefícios e movimentações.

Os dados populacionais utilizados correspondem à posição de dezembro de 2025. Como essa fonte não possui unidade da Federação, os indicadores da previdência fechada representam o conjunto nacional das entidades, não sendo possível atribuí-los diretamente à Região Sudeste.

In [0]:
# Recarrega as tabelas Silver da PREVIC diretamente do catálogo.
# Isso garante o funcionamento mesmo após eventual reinício da sessão.
df_previc_dsi = spark.table(
    "workspace.silver.previc_dsi_2025"
)

df_previc_epb = spark.table(
    "workspace.silver.previc_epb_2025"
)

print("Tabelas Silver da PREVIC carregadas com sucesso.")

Tabelas Silver da PREVIC carregadas com sucesso.


In [0]:
# Mostra a quantidade de registros e entidades por período
# na tabela de informações populacionais.
df_periodos_previc_dsi = (
    df_previc_dsi
    .groupBy("data_referencia", "ano_mes")
    .agg(
        F.count("*").alias("registros"),
        F.countDistinct("codigo_entidade").alias("entidades")
    )
    .orderBy("data_referencia")
)

display(df_periodos_previc_dsi)

data_referencia,ano_mes,registros,entidades
2025-12-01,202512,9938,242


In [0]:
# Identifica as classificações populacionais disponíveis,
# preservando seus códigos e descrições originais.
df_categorias_previc_dsi = (
    df_previc_dsi
    .select(
        "codigo_tipo_populacao",
        "codigo_conta_populacao",
        "descricao_populacao"
    )
    .distinct()
    .orderBy(
        "codigo_tipo_populacao",
        "codigo_conta_populacao"
    )
)

display(df_categorias_previc_dsi)

codigo_tipo_populacao,codigo_conta_populacao,descricao_populacao
19,31000,Participantes Ativos
23,32000,Assistidos - Aposentados
24,33000,Assistidos - Beneficiários de Pensão


In [0]:
# Identifica as categorias de sexo e faixa etária
# presentes nos dados populacionais da PREVIC.
df_perfil_previc_dsi = (
    df_previc_dsi
    .select("sexo", "faixa_etaria")
    .distinct()
    .orderBy("sexo", "faixa_etaria")
)

display(df_perfil_previc_dsi)

sexo,faixa_etaria
F,ATÉ 24 ANOS
F,ENTRE 25 E 34 ANOS
F,ENTRE 35 E 54 ANOS
F,ENTRE 55 E 64 ANOS
F,ENTRE 65 E 74 ANOS
F,ENTRE 75 E 84 ANOS
F,MAIOR QUE 85 ANOS
M,ATÉ 24 ANOS
M,ENTRE 25 E 34 ANOS
M,ENTRE 35 E 54 ANOS


In [0]:
# Resume os registros da tabela de planos por período,
# nível e origem da informação.
df_estrutura_previc_epb = (
    df_previc_epb
    .groupBy(
        "data_referencia",
        "nivel_informacao",
        "origem_informacao"
    )
    .agg(
        F.count("*").alias("registros"),
        F.countDistinct("codigo_entidade").alias("entidades"),
        F.countDistinct("cnpb_plano").alias("planos")
    )
    .orderBy(
        "data_referencia",
        "nivel_informacao",
        "origem_informacao"
    )
)

display(df_estrutura_previc_epb)

data_referencia,nivel_informacao,origem_informacao,registros,entidades,planos
2025-01-01,EFPC,FOLHA,4668,246,0
2025-01-01,EFPC,TOTALIZADOR,489,246,0
2025-01-01,PLANO,FOLHA,20526,246,1087
2025-01-01,PLANO,TOTALIZADOR,2156,246,1087
2025-02-01,EFPC,FOLHA,4642,246,0
2025-02-01,EFPC,TOTALIZADOR,488,246,0
2025-02-01,PLANO,FOLHA,20425,246,1084
2025-02-01,PLANO,TOTALIZADOR,2146,246,1084
2025-03-01,EFPC,FOLHA,4643,246,0
2025-03-01,EFPC,TOTALIZADOR,488,246,0


In [0]:
# Identifica as contas e os tipos de informação
# disponíveis para os planos da PREVIC.
df_contas_previc_epb = (
    df_previc_epb
    .select(
        "codigo_tipo_informacao",
        "codigo_conta",
        "descricao_conta"
    )
    .distinct()
    .orderBy(
        "codigo_tipo_informacao",
        "codigo_conta"
    )
)

display(df_contas_previc_epb)

codigo_tipo_informacao,codigo_conta,descricao_conta
4,11000,Aposentadoria - Prestação Continuada (totalizador)
5,11100,Aposentadoria programada
6,11200,Aposentadoria por Invalidez
7,12000,Auxílios - Prestação Continuada
8,13000,Auxílios - Prestação Única
9,14000,Pensões
10,15000,Pecúlios
11,16000,Outros Benefícios de Prestação Única
12,17000,Outros Benefícios de Prestação Conituada
13,21000,Benefício Proporcional Diferido


### 5.1 Perfil demográfico da previdência complementar fechada

A tabela DSI da PREVIC apresenta a posição de dezembro de 2025 para participantes ativos, aposentados e beneficiários de pensão, classificados por sexo e faixa etária.

Como a fonte não contém unidade da Federação, esse indicador representa o conjunto nacional das entidades de previdência complementar fechada. Essa limitação impede atribuir diretamente os participantes às unidades da Federação da Região Sudeste.

As quantidades consideradas inválidas na camada Silver não são utilizadas no total analítico, mas permanecem contabilizadas em campos de controle e rastreabilidade.

In [0]:
# Padroniza as categorias populacionais e cria uma ordem
# numérica para facilitar a organização das faixas etárias.
df_previc_dsi_base_gold = (
    df_previc_dsi

    # Mantém a posição disponível de dezembro de 2025.
    .filter(F.col("ano_mes") == "202512")

    # Padroniza o tipo de população.
    .withColumn(
        "tipo_populacao",
        F.when(
            F.col("codigo_conta_populacao") == 31000,
            F.lit("PARTICIPANTES_ATIVOS")
        )
        .when(
            F.col("codigo_conta_populacao") == 32000,
            F.lit("APOSENTADOS")
        )
        .when(
            F.col("codigo_conta_populacao") == 33000,
            F.lit("BENEFICIARIOS_PENSAO")
        )
        .otherwise(F.lit("NAO_CLASSIFICADO"))
    )

    # Padroniza as categorias de sexo.
    .withColumn(
        "sexo_padronizado",
        F.when(F.upper(F.trim(F.col("sexo"))) == "F", "FEMININO")
        .when(F.upper(F.trim(F.col("sexo"))) == "M", "MASCULINO")
        .otherwise("NAO_INFORMADO")
    )

    # Cria a ordem lógica das faixas etárias.
    .withColumn(
        "ordem_faixa_etaria",
        F.when(F.col("faixa_etaria") == "ATÉ 24 ANOS", 1)
        .when(F.col("faixa_etaria") == "ENTRE 25 E 34 ANOS", 2)
        .when(F.col("faixa_etaria") == "ENTRE 35 E 54 ANOS", 3)
        .when(F.col("faixa_etaria") == "ENTRE 55 E 64 ANOS", 4)
        .when(F.col("faixa_etaria") == "ENTRE 65 E 74 ANOS", 5)
        .when(F.col("faixa_etaria") == "ENTRE 75 E 84 ANOS", 6)
        .when(
            F.upper(F.col("faixa_etaria")).contains("MAIOR QUE 85"),
            7
        )
    )
)

# Consolida as quantidades de todas as entidades
# por população, sexo e faixa etária.
df_gold_perfil_previdencia_fechada = (
    df_previc_dsi_base_gold
    .groupBy(
        "data_referencia",
        "ano_mes",
        "tipo_populacao",
        "descricao_populacao",
        "sexo_padronizado",
        "faixa_etaria",
        "ordem_faixa_etaria"
    )
    .agg(
        # Soma somente as quantidades analíticas válidas.
        F.sum("quantidade").alias("quantidade_pessoas"),

        # Mantém a soma líquida dos valores informados na fonte,
        # inclusive eventuais ajustes negativos.
        F.sum("quantidade_informada").alias(
            "quantidade_informada_liquida"
        ),

        # Registra quantas linhas foram desconsideradas
        # do total analítico por apresentarem valor inválido.
        F.sum(
            F.when(F.col("flag_quantidade_invalida"), 1).otherwise(0)
        ).alias("registros_quantidade_invalida"),

        # Conta as entidades que forneceram informação em cada grupo.
        F.countDistinct("codigo_entidade").alias(
            "entidades_informantes"
        )
    )
    .withColumn("abrangencia_geografica", F.lit("Brasil"))
    .withColumn("fonte", F.lit("PREVIC - DSI 2025"))
    .withColumn("data_processamento", F.current_timestamp())
    .orderBy(
        "tipo_populacao",
        "sexo_padronizado",
        "ordem_faixa_etaria"
    )
)

display(df_gold_perfil_previdencia_fechada)

data_referencia,ano_mes,tipo_populacao,descricao_populacao,sexo_padronizado,faixa_etaria,ordem_faixa_etaria,quantidade_pessoas,quantidade_informada_liquida,registros_quantidade_invalida,entidades_informantes,abrangencia_geografica,fonte,data_processamento
2025-12-01,202512,APOSENTADOS,Assistidos - Aposentados,FEMININO,ATÉ 24 ANOS,1,114,114,0,227,Brasil,PREVIC - DSI 2025,2026-09-13T02:09:47.464Z
2025-12-01,202512,APOSENTADOS,Assistidos - Aposentados,FEMININO,ENTRE 25 E 34 ANOS,2,191,191,0,229,Brasil,PREVIC - DSI 2025,2026-09-13T02:09:47.464Z
2025-12-01,202512,APOSENTADOS,Assistidos - Aposentados,FEMININO,ENTRE 35 E 54 ANOS,3,6598,6598,0,238,Brasil,PREVIC - DSI 2025,2026-09-13T02:09:47.464Z
2025-12-01,202512,APOSENTADOS,Assistidos - Aposentados,FEMININO,ENTRE 55 E 64 ANOS,4,59856,59856,0,239,Brasil,PREVIC - DSI 2025,2026-09-13T02:09:47.464Z
2025-12-01,202512,APOSENTADOS,Assistidos - Aposentados,FEMININO,ENTRE 65 E 74 ANOS,5,97167,97167,0,240,Brasil,PREVIC - DSI 2025,2026-09-13T02:09:47.464Z
2025-12-01,202512,APOSENTADOS,Assistidos - Aposentados,FEMININO,ENTRE 75 E 84 ANOS,6,39220,39212,1,240,Brasil,PREVIC - DSI 2025,2026-09-13T02:09:47.464Z
2025-12-01,202512,APOSENTADOS,Assistidos - Aposentados,FEMININO,MAIOR QUE 85 ANOS,7,11191,11173,1,237,Brasil,PREVIC - DSI 2025,2026-09-13T02:09:47.464Z
2025-12-01,202512,APOSENTADOS,Assistidos - Aposentados,MASCULINO,ATÉ 24 ANOS,1,144,144,0,227,Brasil,PREVIC - DSI 2025,2026-09-13T02:09:47.464Z
2025-12-01,202512,APOSENTADOS,Assistidos - Aposentados,MASCULINO,ENTRE 25 E 34 ANOS,2,221,221,0,229,Brasil,PREVIC - DSI 2025,2026-09-13T02:09:47.464Z
2025-12-01,202512,APOSENTADOS,Assistidos - Aposentados,MASCULINO,ENTRE 35 E 54 ANOS,3,10962,10962,0,238,Brasil,PREVIC - DSI 2025,2026-09-13T02:09:47.464Z


### 5.2 Participação no perfil demográfico

Para facilitar a interpretação do perfil, é calculada a participação percentual de cada combinação de sexo e faixa etária no total de sua respectiva categoria populacional.

O denominador considera somente quantidades analíticas válidas. Registros sinalizados como inválidos na camada Silver permanecem registrados separadamente para controle.

In [0]:
# Define uma janela para calcular o total de cada
# categoria populacional.
janela_tipo_populacao = Window.partitionBy("tipo_populacao")

# Acrescenta o total da categoria e a participação percentual
# de cada combinação de sexo e faixa etária.
df_gold_perfil_previdencia_fechada = (
    df_gold_perfil_previdencia_fechada

    .withColumn(
        "total_tipo_populacao",
        F.sum("quantidade_pessoas").over(janela_tipo_populacao)
    )

    .withColumn(
        "percentual_dentro_tipo",
        F.when(
            F.col("total_tipo_populacao") > 0,
            (
                F.col("quantidade_pessoas")
                / F.col("total_tipo_populacao")
                * 100
            ).cast("decimal(10,4)")
        )
    )

    .orderBy(
        "tipo_populacao",
        "sexo_padronizado",
        "ordem_faixa_etaria"
    )
)

display(df_gold_perfil_previdencia_fechada)

data_referencia,ano_mes,tipo_populacao,descricao_populacao,sexo_padronizado,faixa_etaria,ordem_faixa_etaria,quantidade_pessoas,quantidade_informada_liquida,registros_quantidade_invalida,entidades_informantes,abrangencia_geografica,fonte,data_processamento,total_tipo_populacao,percentual_dentro_tipo
2025-12-01,202512,APOSENTADOS,Assistidos - Aposentados,FEMININO,ATÉ 24 ANOS,1,114,114,0,227,Brasil,PREVIC - DSI 2025,2026-09-13T02:11:43.352Z,667836,0.0171
2025-12-01,202512,APOSENTADOS,Assistidos - Aposentados,FEMININO,ENTRE 25 E 34 ANOS,2,191,191,0,229,Brasil,PREVIC - DSI 2025,2026-09-13T02:11:43.352Z,667836,0.0286
2025-12-01,202512,APOSENTADOS,Assistidos - Aposentados,FEMININO,ENTRE 35 E 54 ANOS,3,6598,6598,0,238,Brasil,PREVIC - DSI 2025,2026-09-13T02:11:43.352Z,667836,0.9880
2025-12-01,202512,APOSENTADOS,Assistidos - Aposentados,FEMININO,ENTRE 55 E 64 ANOS,4,59856,59856,0,239,Brasil,PREVIC - DSI 2025,2026-09-13T02:11:43.352Z,667836,8.9627
2025-12-01,202512,APOSENTADOS,Assistidos - Aposentados,FEMININO,ENTRE 65 E 74 ANOS,5,97167,97167,0,240,Brasil,PREVIC - DSI 2025,2026-09-13T02:11:43.352Z,667836,14.5495
2025-12-01,202512,APOSENTADOS,Assistidos - Aposentados,FEMININO,ENTRE 75 E 84 ANOS,6,39220,39212,1,240,Brasil,PREVIC - DSI 2025,2026-09-13T02:11:43.352Z,667836,5.8727
2025-12-01,202512,APOSENTADOS,Assistidos - Aposentados,FEMININO,MAIOR QUE 85 ANOS,7,11191,11173,1,237,Brasil,PREVIC - DSI 2025,2026-09-13T02:11:43.352Z,667836,1.6757
2025-12-01,202512,APOSENTADOS,Assistidos - Aposentados,MASCULINO,ATÉ 24 ANOS,1,144,144,0,227,Brasil,PREVIC - DSI 2025,2026-09-13T02:11:43.352Z,667836,0.0216
2025-12-01,202512,APOSENTADOS,Assistidos - Aposentados,MASCULINO,ENTRE 25 E 34 ANOS,2,221,221,0,229,Brasil,PREVIC - DSI 2025,2026-09-13T02:11:43.352Z,667836,0.0331
2025-12-01,202512,APOSENTADOS,Assistidos - Aposentados,MASCULINO,ENTRE 35 E 54 ANOS,3,10962,10962,0,238,Brasil,PREVIC - DSI 2025,2026-09-13T02:11:43.352Z,667836,1.6414


In [0]:
# Calcula os totais diretamente na base Silver utilizada
# como origem da transformação.
df_total_origem_previdencia_fechada = (
    df_previc_dsi_base_gold
    .agg(
        F.sum("quantidade").alias("quantidade_valida_origem"),
        F.sum("quantidade_informada").alias(
            "quantidade_informada_origem"
        ),
        F.sum(
            F.when(F.col("flag_quantidade_invalida"), 1).otherwise(0)
        ).alias("registros_invalidos_origem")
    )
)

# Calcula os mesmos totais na tabela Gold transformada.
df_total_gold_previdencia_fechada = (
    df_gold_perfil_previdencia_fechada
    .agg(
        F.count("*").alias("registros_gold"),
        F.countDistinct("tipo_populacao").alias(
            "tipos_populacao"
        ),
        F.countDistinct("sexo_padronizado").alias(
            "categorias_sexo"
        ),
        F.countDistinct("faixa_etaria").alias(
            "faixas_etarias"
        ),
        F.sum("quantidade_pessoas").alias(
            "quantidade_valida_gold"
        ),
        F.sum("quantidade_informada_liquida").alias(
            "quantidade_informada_gold"
        ),
        F.sum("registros_quantidade_invalida").alias(
            "registros_invalidos_gold"
        )
    )
)

# Reúne os resultados para facilitar a comparação
# entre a origem Silver e a tabela Gold.
df_validacao_perfil_fechada = (
    df_total_gold_previdencia_fechada
    .crossJoin(df_total_origem_previdencia_fechada)
)

display(df_validacao_perfil_fechada)

registros_gold,tipos_populacao,categorias_sexo,faixas_etarias,quantidade_valida_gold,quantidade_informada_gold,registros_invalidos_gold,quantidade_valida_origem,quantidade_informada_origem,registros_invalidos_origem
42,3,2,7,4053330,4053224,12,4053330,4053224,12


In [0]:
# Confere se a soma dos percentuais de cada categoria
# populacional corresponde aproximadamente a 100%.
df_validacao_percentuais_fechada = (
    df_gold_perfil_previdencia_fechada
    .groupBy("tipo_populacao")
    .agg(
        F.sum("quantidade_pessoas").alias("total_pessoas"),
        F.sum("percentual_dentro_tipo").alias(
            "soma_percentual"
        )
    )
    .orderBy("tipo_populacao")
)

display(df_validacao_percentuais_fechada)

tipo_populacao,total_pessoas,soma_percentual
APOSENTADOS,667836,100.0001
BENEFICIARIOS_PENSAO,209579,100.0001
PARTICIPANTES_ATIVOS,3175915,99.9998


### 5.3 Gravação do perfil da previdência fechada

Após a reconciliação das quantidades entre as camadas Silver e Gold, o perfil demográfico é armazenado em formato Delta.

Pequenas diferenças na soma dos percentuais são decorrentes do arredondamento individual para quatro casas decimais e não representam inconsistência nos totais.

In [0]:
# Recupera os resultados consolidados da validação.
validacao_perfil_fechada = (
    df_validacao_perfil_fechada.first()
)

validacao_percentuais_fechada = (
    df_validacao_percentuais_fechada.collect()
)

# Valida a estrutura esperada.
assert validacao_perfil_fechada["registros_gold"] == 42, \
    "Quantidade inesperada de registros."

assert validacao_perfil_fechada["tipos_populacao"] == 3, \
    "Quantidade inesperada de tipos populacionais."

assert validacao_perfil_fechada["categorias_sexo"] == 2, \
    "Quantidade inesperada de categorias de sexo."

assert validacao_perfil_fechada["faixas_etarias"] == 7, \
    "Quantidade inesperada de faixas etárias."

# Confirma que os totais analíticos da Gold coincidem
# exatamente com os totais da tabela Silver de origem.
assert (
    validacao_perfil_fechada["quantidade_valida_gold"]
    == validacao_perfil_fechada["quantidade_valida_origem"]
), "O total válido da Gold diverge da origem."

assert (
    validacao_perfil_fechada["quantidade_informada_gold"]
    == validacao_perfil_fechada["quantidade_informada_origem"]
), "O total informado da Gold diverge da origem."

assert (
    validacao_perfil_fechada["registros_invalidos_gold"]
    == validacao_perfil_fechada["registros_invalidos_origem"]
), "A quantidade de registros inválidos diverge da origem."

# Aceita uma diferença máxima de 0,01 ponto percentual,
# decorrente exclusivamente do arredondamento.
for linha in validacao_percentuais_fechada:
    assert abs(float(linha["soma_percentual"]) - 100) <= 0.01, \
        f"Soma percentual inválida para {linha['tipo_populacao']}."

# Define o nome da tabela Gold de destino.
tabela_gold_perfil_fechada = (
    "workspace.gold.perfil_previdencia_fechada_2025"
)

# Grava a tabela somente após a aprovação das validações.
(
    df_gold_perfil_previdencia_fechada
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(tabela_gold_perfil_fechada)
)

print(
    "Validações aprovadas e tabela Gold gravada com sucesso:",
    tabela_gold_perfil_fechada
)

Validações aprovadas e tabela Gold gravada com sucesso: workspace.gold.perfil_previdencia_fechada_2025


In [0]:
# Lê novamente a tabela diretamente do catálogo Gold.
df_perfil_fechada_gravado = spark.table(
    "workspace.gold.perfil_previdencia_fechada_2025"
)

# Valida a tabela efetivamente armazenada.
df_validacao_perfil_fechada_gravado = (
    df_perfil_fechada_gravado
    .agg(
        F.count("*").alias("quantidade_registros"),
        F.countDistinct("tipo_populacao").alias(
            "tipos_populacao"
        ),
        F.sum("quantidade_pessoas").alias(
            "quantidade_total_pessoas"
        ),
        F.sum("registros_quantidade_invalida").alias(
            "registros_invalidos"
        )
    )
)

display(df_validacao_perfil_fechada_gravado)

quantidade_registros,tipos_populacao,quantidade_total_pessoas,registros_invalidos
42,3,4053330,12


### 5.4 Inspeção das movimentações dos planos no sistema EPB

O Sistema de Captação de Dados de População e Benefícios (EPB) apresenta registros em diferentes níveis de informação:

- `EFPC`: informações consolidadas no nível da entidade fechada;
- `PLANO`: informações individualizadas no nível do plano de benefícios.

Na base analisada também são observadas duas origens de informação:

- `FOLHA`: registros detalhados por conta ou tipo de benefício;
- `TOTALIZADOR`: registros que consolidam determinados conjuntos de contas.

As variáveis quantitativas disponíveis representam:

- `quantidade_inicial`: posição existente no início do período;
- `entradas`: ingressos ou concessões ocorridos durante o período;
- `saidas`: desligamentos, encerramentos ou demais saídas ocorridas no período;
- `quantidade_final`: posição existente ao final do período.

Antes de criar os indicadores Gold, as contas existentes em cada nível e origem são inspecionadas. Essa verificação é necessária para evitar dupla contagem, especialmente quando um totalizador e as contas detalhadas que o compõem aparecem simultaneamente na fonte.

In [0]:
# Identifica o primeiro e o último período da tabela EPB.
df_periodos_previc_epb = (
    df_previc_epb
    .agg(
        F.min("data_referencia").alias("primeira_data"),
        F.max("data_referencia").alias("ultima_data"),
        F.countDistinct("data_referencia").alias(
            "quantidade_periodos"
        )
    )
)

display(df_periodos_previc_epb)

primeira_data,ultima_data,quantidade_periodos
2025-01-01,2025-12-01,12


In [0]:
# Identifica quais contas aparecem em cada origem
# no nível de plano, sem misturar registros de entidade.
df_contas_por_origem_epb = (
    df_previc_epb
    .filter(F.col("nivel_informacao") == "PLANO")
    .groupBy(
        "origem_informacao",
        "codigo_tipo_informacao",
        "codigo_conta",
        "descricao_conta"
    )
    .agg(
        F.count("*").alias("registros"),
        F.countDistinct("cnpb_plano").alias("planos")
    )
    .orderBy(
        "origem_informacao",
        "codigo_conta"
    )
)

display(df_contas_por_origem_epb)

origem_informacao,codigo_tipo_informacao,codigo_conta,descricao_conta,registros,planos
FOLHA,5,11100,Aposentadoria programada,12726,1074
FOLHA,6,11200,Aposentadoria por Invalidez,12591,1062
FOLHA,7,12000,Auxílios - Prestação Continuada,10882,927
FOLHA,8,13000,Auxílios - Prestação Única,10545,899
FOLHA,9,14000,Pensões,12667,1072
FOLHA,10,15000,Pecúlios,12203,1042
FOLHA,11,16000,Outros Benefícios de Prestação Única,11892,1008
FOLHA,12,17000,Outros Benefícios de Prestação Conituada,11730,997
FOLHA,13,21000,Benefício Proporcional Diferido,12616,1063
FOLHA,14,22000,Autopatrocínio,12525,1058


In [0]:
# Obtém dinamicamente a última data disponível.
ultima_data_epb = (
    df_previc_epb
    .agg(F.max("data_referencia").alias("ultima_data"))
    .first()["ultima_data"]
)

# Resume a estrutura do último período no nível de plano.
df_ultimo_periodo_epb = (
    df_previc_epb
    .filter(
        (F.col("data_referencia") == ultima_data_epb)
        & (F.col("nivel_informacao") == "PLANO")
    )
    .groupBy(
        "origem_informacao",
        "codigo_conta",
        "descricao_conta"
    )
    .agg(
        F.count("*").alias("registros"),
        F.countDistinct("codigo_entidade").alias("entidades"),
        F.countDistinct("cnpb_plano").alias("planos"),
        F.sum("quantidade_inicial").alias(
            "quantidade_inicial"
        ),
        F.sum("entradas").alias("entradas"),
        F.sum("saidas").alias("saidas"),
        F.sum("quantidade_final").alias(
            "quantidade_final"
        ),
        F.sum(
            F.when(
                F.col("flag_entrada_invalida")
                | F.col("flag_quantidade_final_invalida"),
                1
            ).otherwise(0)
        ).alias("registros_invalidos")
    )
    .orderBy("origem_informacao", "codigo_conta")
)

print("Última data disponível:", ultima_data_epb)
display(df_ultimo_periodo_epb)

Última data disponível: 2025-12-01


origem_informacao,codigo_conta,descricao_conta,registros,entidades,planos,quantidade_inicial,entradas,saidas,quantidade_final,registros_invalidos
FOLHA,11100,Aposentadoria programada,1060,242,1054,643726,1551,2026,643251,0
FOLHA,11200,Aposentadoria por Invalidez,1049,240,1043,43854,100,235,43719,0
FOLHA,12000,Auxílios - Prestação Continuada,905,214,900,4624,983,992,4615,0
FOLHA,13000,Auxílios - Prestação Única,879,211,874,1151,198,0,1349,0
FOLHA,14000,Pensões,1054,242,1048,198786,1115,743,199158,0
FOLHA,15000,Pecúlios,1019,222,1014,12864,897,0,13761,0
FOLHA,16000,Outros Benefícios de Prestação Única,992,226,986,15160,1167,32,16295,0
FOLHA,17000,Outros Benefícios de Prestação Conituada,977,213,972,364,8,10,362,0
FOLHA,21000,Benefício Proporcional Diferido,1050,235,1044,340296,7992,2656,345632,0
FOLHA,22000,Autopatrocínio,1041,230,1036,46197,459,676,45980,0


### 5.5 Seleção das populações para análise dos planos

Para representar as principais populações dos planos fechados, são selecionadas três contas:

- conta `31000`, de participantes ativos, na origem `TOTALIZADOR`;
- conta `11000`, de aposentadorias de prestação continuada, na origem `TOTALIZADOR`;
- conta `33000`, de beneficiários de pensão, na origem `FOLHA`.

A conta de pensão é utilizada na origem folha porque não existe um registro correspondente na origem totalizador.

Somente registros do nível `PLANO` são considerados. Essa escolha permite analisar as movimentações por plano e evita a duplicação que ocorreria caso os níveis de entidade e plano fossem somados.

In [0]:
# Seleciona as principais populações no nível de plano,
# utilizando totalizadores sempre que eles estão disponíveis.
df_previc_epb_populacoes = (
    df_previc_epb
    .filter(
        (F.col("nivel_informacao") == "PLANO")
        & (
            (
                (F.col("codigo_conta") == 31000)
                & (F.col("origem_informacao") == "TOTALIZADOR")
            )
            |
            (
                (F.col("codigo_conta") == 11000)
                & (F.col("origem_informacao") == "TOTALIZADOR")
            )
            |
            (
                (F.col("codigo_conta") == 33000)
                & (F.col("origem_informacao") == "FOLHA")
            )
        )
    )

    # Cria uma classificação padronizada para cada população.
    .withColumn(
        "tipo_populacao",
        F.when(
            F.col("codigo_conta") == 31000,
            F.lit("PARTICIPANTES_ATIVOS")
        )
        .when(
            F.col("codigo_conta") == 11000,
            F.lit("APOSENTADOS")
        )
        .when(
            F.col("codigo_conta") == 33000,
            F.lit("BENEFICIARIOS_PENSAO")
        )
    )

    # Extrai o ano e o mês da data de referência.
    .withColumn("ano", F.year("data_referencia"))
    .withColumn("mes", F.month("data_referencia"))
)

# Resume a cobertura mensal das contas selecionadas.
df_resumo_populacoes_epb = (
    df_previc_epb_populacoes
    .groupBy(
        "tipo_populacao",
        "codigo_conta",
        "origem_informacao"
    )
    .agg(
        F.count("*").alias("registros"),
        F.countDistinct("codigo_entidade").alias("entidades"),
        F.countDistinct("cnpb_plano").alias("planos"),
        F.countDistinct("data_referencia").alias("periodos"),
        F.min("data_referencia").alias("primeira_data"),
        F.max("data_referencia").alias("ultima_data"),
        F.sum(
            F.when(
                F.col("flag_entrada_invalida")
                | F.col("flag_quantidade_final_invalida"),
                1
            ).otherwise(0)
        ).alias("registros_invalidos")
    )
    .orderBy("tipo_populacao")
)

display(df_resumo_populacoes_epb)

tipo_populacao,codigo_conta,origem_informacao,registros,entidades,planos,periodos,primeira_data,ultima_data,registros_invalidos
APOSENTADOS,11000,TOTALIZADOR,12730,245,1075,12,2025-01-01,2025-12-01,0
BENEFICIARIOS_PENSAO,33000,FOLHA,12746,245,1076,12,2025-01-01,2025-12-01,0
PARTICIPANTES_ATIVOS,31000,TOTALIZADOR,12965,245,1088,12,2025-01-01,2025-12-01,0


### 5.6 Movimentação anual das populações por plano

Para cada plano e categoria populacional, a movimentação de 2025 é consolidada considerando:

- a quantidade inicial do primeiro mês disponível;
- a soma das entradas ocorridas durante o ano;
- a soma das saídas ocorridas durante o ano;
- a quantidade final do último mês disponível.

Também é calculada a diferença entre a variação do estoque e o saldo das movimentações. Essa diferença pode refletir ajustes cadastrais, revisões de informações ou alterações entre competências e não deve ser automaticamente interpretada como erro.

A quantidade de períodos disponíveis é mantida para identificar planos que não possuam os 12 meses completos.

In [0]:
# Consolida previamente eventuais registros de uma mesma
# população, plano e mês, garantindo uma chave mensal única.
df_previc_epb_mensal = (
    df_previc_epb_populacoes
    .groupBy(
        "codigo_entidade",
        "sigla_entidade",
        "codigo_plano",
        "cnpb_plano",
        "tipo_populacao",
        "codigo_conta",
        "ano",
        "mes",
        "data_referencia"
    )
    .agg(
        F.sum("quantidade_inicial").alias("quantidade_inicial_mes"),
        F.sum("entradas").alias("entradas_mes"),
        F.sum("saidas").alias("saidas_mes"),
        F.sum("quantidade_final").alias("quantidade_final_mes"),
        F.sum(
            F.when(
                F.col("flag_entrada_invalida")
                | F.col("flag_quantidade_final_invalida"),
                1
            ).otherwise(0)
        ).alias("registros_invalidos_mes"),
        F.count("*").alias("registros_origem_mes")
    )
)

# Define as janelas para localizar o primeiro e o último
# mês disponível de cada plano e população.
chaves_plano_populacao = [
    "codigo_entidade",
    "sigla_entidade",
    "codigo_plano",
    "cnpb_plano",
    "tipo_populacao",
    "codigo_conta",
    "ano"
]

janela_primeiro_mes_plano = (
    Window
    .partitionBy(*chaves_plano_populacao)
    .orderBy(F.col("data_referencia").asc())
)

janela_ultimo_mes_plano = (
    Window
    .partitionBy(*chaves_plano_populacao)
    .orderBy(F.col("data_referencia").desc())
)

# Obtém a quantidade inicial do primeiro mês disponível.
df_previc_epb_posicao_inicial = (
    df_previc_epb_mensal
    .withColumn(
        "ordem_primeiro_mes",
        F.row_number().over(janela_primeiro_mes_plano)
    )
    .filter(F.col("ordem_primeiro_mes") == 1)
    .select(
        *chaves_plano_populacao,
        F.col("mes").alias("primeiro_mes_disponivel"),
        F.col("data_referencia").alias("data_posicao_inicial"),
        F.col("quantidade_inicial_mes").alias(
            "quantidade_inicial_ano"
        )
    )
)

# Obtém a quantidade final do último mês disponível.
df_previc_epb_posicao_final = (
    df_previc_epb_mensal
    .withColumn(
        "ordem_ultimo_mes",
        F.row_number().over(janela_ultimo_mes_plano)
    )
    .filter(F.col("ordem_ultimo_mes") == 1)
    .select(
        *chaves_plano_populacao,
        F.col("mes").alias("ultimo_mes_disponivel"),
        F.col("data_referencia").alias("data_posicao_final"),
        F.col("quantidade_final_mes").alias(
            "quantidade_final_ano"
        )
    )
)

# Soma as entradas e saídas mensais de cada plano no ano.
df_previc_epb_fluxos_anuais = (
    df_previc_epb_mensal
    .groupBy(*chaves_plano_populacao)
    .agg(
        F.sum("entradas_mes").alias("entradas_ano"),
        F.sum("saidas_mes").alias("saidas_ano"),
        F.countDistinct("mes").alias("periodos_com_dados"),
        F.sum("registros_invalidos_mes").alias(
            "registros_invalidos"
        ),
        F.sum("registros_origem_mes").alias(
            "registros_origem"
        )
    )
)

# Combina posição inicial, movimentações anuais e posição final.
df_gold_movimentacao_planos_fechados = (
    df_previc_epb_fluxos_anuais.alias("fluxos")
    .join(
        df_previc_epb_posicao_inicial.alias("inicial"),
        on=chaves_plano_populacao,
        how="inner"
    )
    .join(
        df_previc_epb_posicao_final.alias("final"),
        on=chaves_plano_populacao,
        how="inner"
    )
    .select(
        *chaves_plano_populacao,
        "primeiro_mes_disponivel",
        "ultimo_mes_disponivel",
        "data_posicao_inicial",
        "data_posicao_final",
        "periodos_com_dados",
        "quantidade_inicial_ano",
        "entradas_ano",
        "saidas_ano",
        "quantidade_final_ano",
        "registros_invalidos",
        "registros_origem"
    )

    # Calcula o saldo líquido das movimentações.
    .withColumn(
        "saldo_movimentacao_ano",
        F.col("entradas_ano") - F.col("saidas_ano")
    )

    # Calcula a variação observada entre a posição
    # inicial e a posição final.
    .withColumn(
        "variacao_estoque_ano",
        F.col("quantidade_final_ano")
        - F.col("quantidade_inicial_ano")
    )

    # Mede diferenças não explicadas somente pelas
    # entradas e saídas acumuladas.
    .withColumn(
        "diferenca_reconciliacao",
        F.col("variacao_estoque_ano")
        - F.col("saldo_movimentacao_ano")
    )

    .withColumn("abrangencia_geografica", F.lit("Brasil"))
    .withColumn("fonte", F.lit("PREVIC - EPB 2025"))
    .withColumn("data_processamento", F.current_timestamp())
    .orderBy(
        "tipo_populacao",
        "codigo_entidade",
        "cnpb_plano"
    )
)

display(df_gold_movimentacao_planos_fechados)

codigo_entidade,sigla_entidade,codigo_plano,cnpb_plano,tipo_populacao,codigo_conta,ano,primeiro_mes_disponivel,ultimo_mes_disponivel,data_posicao_inicial,data_posicao_final,periodos_com_dados,quantidade_inicial_ano,entradas_ano,saidas_ano,quantidade_final_ano,registros_invalidos,registros_origem,saldo_movimentacao_ano,variacao_estoque_ano,diferenca_reconciliacao,abrangencia_geografica,fonte,data_processamento
14,AGROS,13,1980000883,APOSENTADOS,11000,2025,1,12,2025-01-01,2025-12-01,12,70,0,6,64,0,12,-6,-6,0,Brasil,PREVIC - EPB 2025,2026-09-13T02:50:46.477Z
14,AGROS,14,1992000174,APOSENTADOS,11000,2025,1,12,2025-01-01,2025-12-01,12,0,0,0,0,0,12,0,0,0,Brasil,PREVIC - EPB 2025,2026-09-13T02:50:46.477Z
14,AGROS,1157,2008001083,APOSENTADOS,11000,2025,1,12,2025-01-01,2025-12-01,12,29,3,0,32,0,12,3,3,0,Brasil,PREVIC - EPB 2025,2026-09-13T02:50:46.477Z
14,AGROS,8687,2023001692,APOSENTADOS,11000,2025,1,12,2025-01-01,2025-12-01,12,2773,40,63,2750,0,12,-23,-23,0,Brasil,PREVIC - EPB 2025,2026-09-13T02:50:46.477Z
31,ALCOA PREVI,15,1988003156,APOSENTADOS,11000,2025,1,12,2025-01-01,2025-12-01,12,178,10,4,184,0,12,6,6,0,Brasil,PREVIC - EPB 2025,2026-09-13T02:50:46.477Z
45,ALPHA,16,1999002474,APOSENTADOS,11000,2025,1,12,2025-01-01,2025-12-01,12,197,18,5,210,0,12,13,13,0,Brasil,PREVIC - EPB 2025,2026-09-13T02:50:46.477Z
59,INFRAPREV,18,1982000783,APOSENTADOS,11000,2025,1,12,2025-01-01,2025-12-01,12,85,0,5,80,0,12,-5,-5,0,Brasil,PREVIC - EPB 2025,2026-09-13T02:50:46.477Z
59,INFRAPREV,19,1982001811,APOSENTADOS,11000,2025,1,12,2025-01-01,2025-12-01,12,9,0,1,8,0,12,-1,-1,0,Brasil,PREVIC - EPB 2025,2026-09-13T02:50:46.477Z
59,INFRAPREV,17,2000008119,APOSENTADOS,11000,2025,1,12,2025-01-01,2025-12-01,12,3809,421,276,3954,0,12,145,145,0,Brasil,PREVIC - EPB 2025,2026-09-13T02:50:46.477Z
59,INFRAPREV,3704,2012000819,APOSENTADOS,11000,2025,1,12,2025-01-01,2025-12-01,12,194,138,1,331,0,12,137,137,0,Brasil,PREVIC - EPB 2025,2026-09-13T02:50:46.477Z


### 5.7 Validação das movimentações anuais dos planos

A tabela é validada quanto à quantidade de registros, populações, entidades, planos, valores inválidos, identificadores ausentes, duplicidades e cobertura mensal.

Também é verificada a reconciliação entre a variação do estoque e o saldo de entradas e saídas. Planos com menos de 12 períodos são preservados e identificados, pois podem representar início, encerramento ou ausência de informação em determinadas competências.


In [0]:
# Calcula os principais indicadores de qualidade
# da movimentação anual dos planos.
df_validacao_movimentacao_fechada = (
    df_gold_movimentacao_planos_fechados
    .agg(
        F.count("*").alias("registros"),
        F.countDistinct("tipo_populacao").alias(
            "tipos_populacao"
        ),
        F.countDistinct("codigo_entidade").alias("entidades"),
        F.countDistinct("cnpb_plano").alias("planos"),

        # Identifica planos sem os principais códigos.
        F.sum(
            F.when(F.col("cnpb_plano").isNull(), 1).otherwise(0)
        ).alias("cnpb_plano_nulo"),
        F.sum(
            F.when(F.col("codigo_plano").isNull(), 1).otherwise(0)
        ).alias("codigo_plano_nulo"),

        # Confere valores analíticos ausentes.
        F.sum(
            F.when(
                F.col("quantidade_inicial_ano").isNull(),
                1
            ).otherwise(0)
        ).alias("quantidades_iniciais_nulas"),
        F.sum(
            F.when(
                F.col("quantidade_final_ano").isNull(),
                1
            ).otherwise(0)
        ).alias("quantidades_finais_nulas"),

        # Consolida os registros inválidos preservados.
        F.sum("registros_invalidos").alias(
            "registros_invalidos"
        ),

        # Identifica diferenças na reconciliação.
        F.sum(
            F.when(
                F.col("diferenca_reconciliacao") != 0,
                1
            ).otherwise(0)
        ).alias("registros_com_diferenca"),

        F.max(
            F.abs(F.col("diferenca_reconciliacao"))
        ).alias("maior_diferenca_absoluta")
    )
)

display(df_validacao_movimentacao_fechada)

registros,tipos_populacao,entidades,planos,cnpb_plano_nulo,codigo_plano_nulo,quantidades_iniciais_nulas,quantidades_finais_nulas,registros_invalidos,registros_com_diferenca,maior_diferenca_absoluta
3266,3,246,1091,0,0,0,0,0,28,105


In [0]:
# Mostra quantos registros possuem de 1 a 12 meses,
# permitindo identificar planos com cobertura incompleta.
df_cobertura_movimentacao_fechada = (
    df_gold_movimentacao_planos_fechados
    .groupBy("periodos_com_dados")
    .agg(
        F.count("*").alias("registros"),
        F.countDistinct("cnpb_plano").alias("planos")
    )
    .orderBy("periodos_com_dados")
)

display(df_cobertura_movimentacao_fechada)

periodos_com_dados,registros,planos
1,17,10
2,4,4
3,6,4
4,7,4
5,14,6
6,42,17
7,12,4
8,6,3
9,1,1
10,1,1


In [0]:
# Verifica se existe mais de uma linha para a mesma
# combinação de plano, população e ano.
df_repeticoes_movimentacao_fechada = (
    df_gold_movimentacao_planos_fechados
    .groupBy("cnpb_plano", "tipo_populacao", "ano")
    .count()
    .filter(F.col("count") > 1)
    .agg(
        F.count("*").alias("grupos_repetidos"),
        F.sum(F.col("count") - 1).alias("linhas_excedentes"),
        F.max("count").alias("maior_repeticao")
    )
)

display(df_repeticoes_movimentacao_fechada)

grupos_repetidos,linhas_excedentes,maior_repeticao
30,30,2


### 5.8 Diagnóstico de chaves repetidas e diferenças de reconciliação

A validação identificou situações em que o mesmo CNPB e população aparecem em mais de uma linha anual. O diagnóstico verifica se essas ocorrências são causadas por mudanças no código do plano, código da entidade ou sigla da entidade durante o período.

A tabela não será gravada antes da resolução dessas chaves, evitando duplicidade nos indicadores por plano.

In [0]:
# Identifica as chaves anuais que aparecem mais de uma vez.
df_chaves_repetidas_fechada = (
    df_gold_movimentacao_planos_fechados
    .groupBy("cnpb_plano", "tipo_populacao", "ano")
    .agg(
        F.count("*").alias("quantidade_linhas"),
        F.countDistinct("codigo_entidade").alias(
            "codigos_entidade_distintos"
        ),
        F.countDistinct("sigla_entidade").alias(
            "siglas_entidade_distintas"
        ),
        F.countDistinct("codigo_plano").alias(
            "codigos_plano_distintos"
        ),
        F.collect_set("codigo_entidade").alias(
            "codigos_entidade"
        ),
        F.collect_set("sigla_entidade").alias(
            "siglas_entidade"
        ),
        F.collect_set("codigo_plano").alias(
            "codigos_plano"
        )
    )
    .filter(F.col("quantidade_linhas") > 1)
    .orderBy("cnpb_plano", "tipo_populacao")
)

display(df_chaves_repetidas_fechada)

cnpb_plano,tipo_populacao,ano,quantidade_linhas,codigos_entidade_distintos,siglas_entidade_distintas,codigos_plano_distintos,codigos_entidade,siglas_entidade,codigos_plano
1974000338,APOSENTADOS,2025,2,2,2,1,"List(2525, 4091)","List(ELOS, PREVIG)",List(604)
1974000338,BENEFICIARIOS_PENSAO,2025,2,2,2,1,"List(2525, 4091)","List(ELOS, PREVIG)",List(604)
1974000338,PARTICIPANTES_ATIVOS,2025,2,2,2,1,"List(2525, 4091)","List(ELOS, PREVIG)",List(604)
1988000947,APOSENTADOS,2025,2,2,2,1,"List(3126, 4443)","List(IFM, ENERPREV)",List(74)
1988000947,BENEFICIARIOS_PENSAO,2025,2,2,2,1,"List(3126, 4443)","List(IFM, ENERPREV)",List(74)
1988000947,PARTICIPANTES_ATIVOS,2025,2,2,2,1,"List(3126, 4443)","List(IFM, ENERPREV)",List(74)
1990002129,APOSENTADOS,2025,2,2,2,1,"List(2292, 3825)","List(FUNSSEST, MULTIPENSIONS)",List(943)
1990002129,BENEFICIARIOS_PENSAO,2025,2,2,2,1,"List(2292, 3825)","List(FUNSSEST, MULTIPENSIONS)",List(943)
1990002129,PARTICIPANTES_ATIVOS,2025,2,2,2,1,"List(2292, 3825)","List(FUNSSEST, MULTIPENSIONS)",List(943)
1996004247,APOSENTADOS,2025,2,2,2,1,"List(3126, 3575)","List(IFM, CARBOPREV)",List(776)


In [0]:
# Recupera todas as linhas da tabela anual relacionadas
# às chaves repetidas para comparar seus identificadores.
df_detalhes_repeticoes_fechada = (
    df_gold_movimentacao_planos_fechados.alias("dados")
    .join(
        df_chaves_repetidas_fechada
        .select("cnpb_plano", "tipo_populacao", "ano")
        .alias("repeticoes"),
        on=["cnpb_plano", "tipo_populacao", "ano"],
        how="inner"
    )
    .select(
        "cnpb_plano",
        "tipo_populacao",
        "ano",
        "codigo_entidade",
        "sigla_entidade",
        "codigo_plano",
        "primeiro_mes_disponivel",
        "ultimo_mes_disponivel",
        "periodos_com_dados",
        "quantidade_inicial_ano",
        "entradas_ano",
        "saidas_ano",
        "quantidade_final_ano",
        "diferenca_reconciliacao"
    )
    .orderBy(
        "cnpb_plano",
        "tipo_populacao",
        "primeiro_mes_disponivel"
    )
)

display(df_detalhes_repeticoes_fechada)

cnpb_plano,tipo_populacao,ano,codigo_entidade,sigla_entidade,codigo_plano,primeiro_mes_disponivel,ultimo_mes_disponivel,periodos_com_dados,quantidade_inicial_ano,entradas_ano,saidas_ano,quantidade_final_ano,diferenca_reconciliacao
1974000338,APOSENTADOS,2025,4091,PREVIG,604,1,12,12,1191,0,39,1152,0
1974000338,APOSENTADOS,2025,2525,ELOS,604,1,5,5,0,0,0,0,0
1974000338,BENEFICIARIOS_PENSAO,2025,2525,ELOS,604,1,5,5,0,0,0,0,0
1974000338,BENEFICIARIOS_PENSAO,2025,4091,PREVIG,604,1,12,12,674,25,23,676,0
1974000338,PARTICIPANTES_ATIVOS,2025,4091,PREVIG,604,1,12,12,0,0,0,0,0
1974000338,PARTICIPANTES_ATIVOS,2025,2525,ELOS,604,1,5,5,0,0,0,0,0
1988000947,APOSENTADOS,2025,4443,ENERPREV,74,1,7,7,317,0,317,0,0
1988000947,APOSENTADOS,2025,3126,IFM,74,8,12,5,0,302,5,297,0
1988000947,BENEFICIARIOS_PENSAO,2025,4443,ENERPREV,74,1,7,7,283,14,297,0,0
1988000947,BENEFICIARIOS_PENSAO,2025,3126,IFM,74,8,12,5,0,292,4,288,0


In [0]:
# Exibe somente os registros cuja variação do estoque
# não foi totalmente explicada pelas entradas e saídas.
df_diferencas_reconciliacao_fechada = (
    df_gold_movimentacao_planos_fechados
    .filter(F.col("diferenca_reconciliacao") != 0)
    .select(
        "cnpb_plano",
        "tipo_populacao",
        "codigo_entidade",
        "sigla_entidade",
        "codigo_plano",
        "periodos_com_dados",
        "quantidade_inicial_ano",
        "entradas_ano",
        "saidas_ano",
        "quantidade_final_ano",
        "diferenca_reconciliacao"
    )
    .orderBy(
        F.abs(F.col("diferenca_reconciliacao")).desc()
    )
)

display(df_diferencas_reconciliacao_fechada)

cnpb_plano,tipo_populacao,codigo_entidade,sigla_entidade,codigo_plano,periodos_com_dados,quantidade_inicial_ano,entradas_ano,saidas_ano,quantidade_final_ano,diferenca_reconciliacao
2015001174,PARTICIPANTES_ATIVOS,2796,PORTOPREV,4864,11,7214,1617,880,8056,105
1997002329,PARTICIPANTES_ATIVOS,211,CELOS,49,12,3877,252,188,3890,-51
2018002392,APOSENTADOS,4176,SEBRAE PREVIDENCIA,6045,12,0,50,0,0,-50
1996003038,PARTICIPANTES_ATIVOS,3531,RBS PREV,771,12,5683,1678,2025,5358,22
1993002529,PARTICIPANTES_ATIVOS,2796,PORTOPREV,625,11,3273,122,299,3079,-17
2005004211,PARTICIPANTES_ATIVOS,4368,OABPREV-SP,1046,11,51566,2709,2575,51717,17
2009000919,PARTICIPANTES_ATIVOS,4546,DATUSPREV,1692,6,298,0,45,246,-7
1979001618,BENEFICIARIOS_PENSAO,1451,FIPECQ,309,12,117,10,3,130,6
2005004211,APOSENTADOS,4368,OABPREV-SP,1046,11,274,43,35,286,4
1995002992,APOSENTADOS,3321,TRAMONTINAPREV,746,12,99,8,13,96,2


### 5.9 Diagnóstico de transferência de gestão entre entidades

Foram identificados planos que mantiveram o mesmo CNPB e código de plano, mas apareceram associados a diferentes entidades durante 2025.

Essa situação pode representar transferência de gestão, incorporação ou alteração cadastral. Como o CNPB identifica o plano de forma estável, esses registros não devem produzir mais de uma linha anual.

Antes da consolidação, são investigados os meses em que mais de uma entidade informou dados para o mesmo plano e população, evitando a soma indevida de informações sobrepostas.

In [0]:
# Identifica meses em que o mesmo CNPB e população
# aparecem associados a mais de uma entidade.
df_sobreposicoes_mensais_epb = (
    df_previc_epb_populacoes
    .groupBy(
        "cnpb_plano",
        "tipo_populacao",
        "ano",
        "mes",
        "data_referencia"
    )
    .agg(
        F.countDistinct("codigo_entidade").alias(
            "entidades_no_mes"
        ),
        F.collect_set("sigla_entidade").alias(
            "siglas_entidade"
        ),

        # Conta quantas entidades apresentaram algum estoque
        # inicial ou final diferente de zero.
        F.sum(
            F.when(
                (F.coalesce(F.col("quantidade_inicial"), F.lit(0)) != 0)
                | (F.coalesce(F.col("quantidade_final"), F.lit(0)) != 0),
                1
            ).otherwise(0)
        ).alias("linhas_com_estoque_nao_zero"),

        # Compara soma e maior valor para avaliar
        # se os registros são complementares ou duplicados.
        F.sum("quantidade_inicial").alias(
            "soma_quantidade_inicial"
        ),
        F.max("quantidade_inicial").alias(
            "maior_quantidade_inicial"
        ),
        F.sum("quantidade_final").alias(
            "soma_quantidade_final"
        ),
        F.max("quantidade_final").alias(
            "maior_quantidade_final"
        )
    )
    .filter(F.col("entidades_no_mes") > 1)
    .orderBy(
        "cnpb_plano",
        "tipo_populacao",
        "data_referencia"
    )
)

display(df_sobreposicoes_mensais_epb)

cnpb_plano,tipo_populacao,ano,mes,data_referencia,entidades_no_mes,siglas_entidade,linhas_com_estoque_nao_zero,soma_quantidade_inicial,maior_quantidade_inicial,soma_quantidade_final,maior_quantidade_final
1974000338,APOSENTADOS,2025,1,2025-01-01,2,"List(PREVIG, ELOS)",1,1191,1191,1189,1189
1974000338,APOSENTADOS,2025,2,2025-02-01,2,"List(PREVIG, ELOS)",1,1189,1189,1187,1187
1974000338,APOSENTADOS,2025,3,2025-03-01,2,"List(ELOS, PREVIG)",1,1187,1187,1184,1184
1974000338,APOSENTADOS,2025,4,2025-04-01,2,"List(PREVIG, ELOS)",1,1184,1184,1183,1183
1974000338,APOSENTADOS,2025,5,2025-05-01,2,"List(ELOS, PREVIG)",1,1183,1183,1182,1182
1974000338,BENEFICIARIOS_PENSAO,2025,1,2025-01-01,2,"List(PREVIG, ELOS)",1,674,674,675,675
1974000338,BENEFICIARIOS_PENSAO,2025,2,2025-02-01,2,"List(PREVIG, ELOS)",1,675,675,677,677
1974000338,BENEFICIARIOS_PENSAO,2025,3,2025-03-01,2,"List(ELOS, PREVIG)",1,677,677,678,678
1974000338,BENEFICIARIOS_PENSAO,2025,4,2025-04-01,2,"List(PREVIG, ELOS)",1,678,678,680,680
1974000338,BENEFICIARIOS_PENSAO,2025,5,2025-05-01,2,"List(PREVIG, ELOS)",1,680,680,682,682


In [0]:
# Resume a quantidade e a natureza das sobreposições.
df_resumo_sobreposicoes_epb = (
    df_sobreposicoes_mensais_epb
    .agg(
        F.count("*").alias("meses_com_mais_de_uma_entidade"),

        F.countDistinct("cnpb_plano").alias(
            "planos_com_sobreposicao"
        ),

        # Mais de uma linha não zero indica possível
        # sobreposição efetiva dos estoques.
        F.sum(
            F.when(
                F.col("linhas_com_estoque_nao_zero") > 1,
                1
            ).otherwise(0)
        ).alias("meses_com_multiplos_estoques_nao_zero"),

        # Soma diferente do maior valor indica que mais
        # de um registro contribuiu para o estoque do mês.
        F.sum(
            F.when(
                F.col("soma_quantidade_final")
                != F.col("maior_quantidade_final"),
                1
            ).otherwise(0)
        ).alias("meses_em_que_soma_difere_do_maior")
    )
)

display(df_resumo_sobreposicoes_epb)

meses_com_mais_de_uma_entidade,planos_com_sobreposicao,meses_com_multiplos_estoques_nao_zero,meses_em_que_soma_difere_do_maior
186,7,12,5


### 5.10 Análise das sobreposições efetivas

A maior parte das ocorrências com mais de uma entidade apresenta estoque em apenas um dos registros, indicando manutenção simultânea do cadastro durante uma transferência de gestão.

Foram identificados cinco casos em que a soma dos estoques finais difere do maior valor individual. Esses casos são analisados detalhadamente antes da definição da regra de consolidação.

In [0]:
# Seleciona somente os cinco meses em que mais de uma
# entidade contribuiu para o estoque final do mesmo plano.
df_chaves_sobreposicao_efetiva = (
    df_sobreposicoes_mensais_epb
    .filter(
        F.col("soma_quantidade_final")
        != F.col("maior_quantidade_final")
    )
    .select(
        "cnpb_plano",
        "tipo_populacao",
        "data_referencia"
    )
)

# Recupera os registros individuais das entidades
# envolvidos nas cinco sobreposições efetivas.
df_detalhes_sobreposicao_efetiva = (
    df_previc_epb_populacoes.alias("dados")
    .join(
        df_chaves_sobreposicao_efetiva.alias("chaves"),
        on=[
            "cnpb_plano",
            "tipo_populacao",
            "data_referencia"
        ],
        how="inner"
    )
    .select(
        "cnpb_plano",
        "tipo_populacao",
        "data_referencia",
        "codigo_entidade",
        "sigla_entidade",
        "codigo_plano",
        "quantidade_inicial",
        "entradas",
        "saidas",
        "quantidade_final",
        "flag_entrada_invalida",
        "flag_quantidade_final_invalida"
    )
    .orderBy(
        "cnpb_plano",
        "tipo_populacao",
        "data_referencia",
        F.col("quantidade_final").desc()
    )
)

display(df_detalhes_sobreposicao_efetiva)

cnpb_plano,tipo_populacao,data_referencia,codigo_entidade,sigla_entidade,codigo_plano,quantidade_inicial,entradas,saidas,quantidade_final,flag_entrada_invalida,flag_quantidade_final_invalida
2002000492,APOSENTADOS,2025-07-01,3825,MULTIPENSIONS,116,0,12,0,12,false,false
2002000492,APOSENTADOS,2025-07-01,655,PETROS,116,12,0,0,12,false,false
2002000492,PARTICIPANTES_ATIVOS,2025-07-01,3825,MULTIPENSIONS,116,0,1188,0,1188,false,false
2002000492,PARTICIPANTES_ATIVOS,2025-07-01,655,PETROS,116,404,0,0,404,false,false
2007002965,PARTICIPANTES_ATIVOS,2025-01-01,4683,VIVA,1135,0,180,0,180,false,false
2007002965,PARTICIPANTES_ATIVOS,2025-01-01,775,MULTIBRA INSTITUIDOR,1135,14,0,2,12,false,false
2007002965,PARTICIPANTES_ATIVOS,2025-02-01,4683,VIVA,1135,180,0,0,180,false,false
2007002965,PARTICIPANTES_ATIVOS,2025-02-01,775,MULTIBRA INSTITUIDOR,1135,12,0,0,12,false,false
2007002965,PARTICIPANTES_ATIVOS,2025-03-01,4683,VIVA,1135,180,0,1,179,false,false
2007002965,PARTICIPANTES_ATIVOS,2025-03-01,775,MULTIBRA INSTITUIDOR,1135,12,1,2,11,false,false


### 5.11 Tratamento das mudanças de entidade gestora

A análise identificou planos que mantiveram o mesmo CNPB, mas apareceram associados a mais de uma entidade durante 2025.

Em alguns meses, a nova entidade registra entradas enquanto a entidade anterior ainda mantém parte ou todo o estoque. A fonte não permite determinar com segurança se os valores representam duplicação temporária, transferência integral ou manutenção de parcelas residuais.

Para preservar a informação original e evitar uma regra arbitrária de soma ou exclusão, a tabela Gold mantém a granularidade de plano e entidade. São criados indicadores para identificar:

- planos associados a mais de uma entidade no ano;
- registros com menos de 12 competências;
- diferenças entre a variação do estoque e o saldo das movimentações;
- registros aptos para análises comparáveis de movimentação.

Assim, os casos ambíguos permanecem disponíveis para auditoria, mas podem ser separados das análises consolidadas.

In [0]:
# Identifica os CNPBs associados a mais de uma entidade
# durante o ano, independentemente da população.
df_planos_com_mudanca_entidade = (
    df_previc_epb_populacoes
    .groupBy("cnpb_plano", "ano")
    .agg(
        F.countDistinct("codigo_entidade").alias(
            "quantidade_entidades_no_ano"
        ),
        F.collect_set("sigla_entidade").alias(
            "entidades_identificadas"
        )
    )
    .filter(F.col("quantidade_entidades_no_ano") > 1)
    .withColumn("flag_mudanca_entidade", F.lit(True))
)

# Acrescenta os indicadores de qualidade à tabela anual,
# mantendo separadas as combinações de plano e entidade.
df_gold_movimentacao_planos_fechados = (
    df_gold_movimentacao_planos_fechados.alias("movimentacao")
    .join(
        df_planos_com_mudanca_entidade.alias("mudanca"),
        on=["cnpb_plano", "ano"],
        how="left"
    )

    # Planos não encontrados na tabela de mudança
    # permanecem com indicador falso.
    .withColumn(
        "flag_mudanca_entidade",
        F.coalesce(
            F.col("flag_mudanca_entidade"),
            F.lit(False)
        )
    )

    # Identifica registros que não possuem os 12 meses.
    .withColumn(
        "flag_cobertura_incompleta",
        F.col("periodos_com_dados") < 12
    )

    # Identifica diferenças não explicadas exclusivamente
    # pelas entradas e saídas informadas.
    .withColumn(
        "flag_diferenca_reconciliacao",
        F.col("diferenca_reconciliacao") != 0
    )

    # Define como aptos para comparação os registros completos,
    # válidos, sem mudança de entidade e reconciliados.
    .withColumn(
        "flag_apto_analise_movimentacao",
        (F.col("periodos_com_dados") == 12)
        & (F.col("registros_invalidos") == 0)
        & (~F.col("flag_mudanca_entidade"))
        & (F.col("diferenca_reconciliacao") == 0)
    )

    .orderBy(
        "tipo_populacao",
        "codigo_entidade",
        "cnpb_plano"
    )
)

# Resume os indicadores criados.
df_resumo_qualidade_movimentacao_fechada = (
    df_gold_movimentacao_planos_fechados
    .agg(
        F.count("*").alias("registros"),
        F.countDistinct("cnpb_plano").alias("planos"),
        F.countDistinct(
            F.when(
                F.col("flag_mudanca_entidade"),
                F.col("cnpb_plano")
            )
        ).alias("planos_com_mudanca_entidade"),
        F.sum(
            F.when(
                F.col("flag_cobertura_incompleta"),
                1
            ).otherwise(0)
        ).alias("registros_cobertura_incompleta"),
        F.sum(
            F.when(
                F.col("flag_diferenca_reconciliacao"),
                1
            ).otherwise(0)
        ).alias("registros_com_diferenca"),
        F.sum(
            F.when(
                F.col("flag_apto_analise_movimentacao"),
                1
            ).otherwise(0)
        ).alias("registros_aptos_analise")
    )
)

display(df_resumo_qualidade_movimentacao_fechada)

registros,planos,planos_com_mudanca_entidade,registros_cobertura_incompleta,registros_com_diferenca,registros_aptos_analise
3266,1091,10,121,28,3092


In [0]:
# Verifica a unicidade por plano, entidade,
# população e ano.
df_repeticoes_chave_real_fechada = (
    df_gold_movimentacao_planos_fechados
    .groupBy(
        "cnpb_plano",
        "codigo_entidade",
        "tipo_populacao",
        "ano"
    )
    .count()
    .filter(F.col("count") > 1)
    .agg(
        F.count("*").alias("grupos_repetidos"),
        F.sum(F.col("count") - 1).alias("linhas_excedentes"),
        F.max("count").alias("maior_repeticao")
    )
)

display(df_repeticoes_chave_real_fechada)

grupos_repetidos,linhas_excedentes,maior_repeticao
0,null,null


### 5.12 Gravação das movimentações dos planos e entidades

A tabela é armazenada mantendo a granularidade de plano, entidade, população e ano.

Os registros com mudança de entidade, cobertura incompleta ou diferença de reconciliação não são apagados. Eles permanecem acompanhados de indicadores de qualidade, permitindo rastreabilidade e seleção segura nas análises posteriores.

O campo `flag_apto_analise_movimentacao` identifica os registros que apresentam 12 meses, não possuem valores inválidos, não apresentam mudança de entidade e possuem movimentações reconciliadas.

In [0]:
# Recupera os resultados das validações anteriores.
validacao_movimentacao_fechada = (
    df_resumo_qualidade_movimentacao_fechada.first()
)

validacao_chave_real_fechada = (
    df_repeticoes_chave_real_fechada.first()
)

# Confirma os resultados esperados.
assert validacao_movimentacao_fechada["registros"] == 3266, \
    "Quantidade inesperada de registros."

assert validacao_movimentacao_fechada["planos"] == 1091, \
    "Quantidade inesperada de planos."

assert (
    validacao_movimentacao_fechada["planos_com_mudanca_entidade"]
    == 10
), "Quantidade inesperada de planos com mudança de entidade."

assert (
    validacao_movimentacao_fechada["registros_cobertura_incompleta"]
    == 121
), "Quantidade inesperada de coberturas incompletas."

assert (
    validacao_movimentacao_fechada["registros_com_diferenca"]
    == 28
), "Quantidade inesperada de diferenças de reconciliação."

assert (
    validacao_movimentacao_fechada["registros_aptos_analise"]
    == 3092
), "Quantidade inesperada de registros aptos."

assert validacao_chave_real_fechada["grupos_repetidos"] == 0, \
    "Foram encontradas duplicidades na chave real."

# Define o nome da tabela conforme sua granularidade.
tabela_gold_movimentacao_fechada = (
    "workspace.gold.movimentacao_plano_entidade_fechada_2025"
)

# Grava todos os registros e seus indicadores de qualidade.
(
    df_gold_movimentacao_planos_fechados
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(tabela_gold_movimentacao_fechada)
)

print(
    "Validações aprovadas e tabela Gold gravada com sucesso:",
    tabela_gold_movimentacao_fechada
)

Validações aprovadas e tabela Gold gravada com sucesso: workspace.gold.movimentacao_plano_entidade_fechada_2025


In [0]:
# Lê novamente a tabela diretamente do catálogo Gold.
df_movimentacao_fechada_gravado = spark.table(
    "workspace.gold.movimentacao_plano_entidade_fechada_2025"
)

# Confirma a quantidade total e os registros aptos.
df_validacao_movimentacao_gravado = (
    df_movimentacao_fechada_gravado
    .agg(
        F.count("*").alias("quantidade_registros"),
        F.countDistinct("cnpb_plano").alias("planos"),
        F.sum(
            F.when(
                F.col("flag_apto_analise_movimentacao"),
                1
            ).otherwise(0)
        ).alias("registros_aptos"),
        F.sum(
            F.when(
                F.col("flag_mudanca_entidade"),
                1
            ).otherwise(0)
        ).alias("linhas_com_mudanca_entidade")
    )
)

display(df_validacao_movimentacao_gravado)

quantidade_registros,planos,registros_aptos,linhas_com_mudanca_entidade
3266,1091,3092,60


## 6. Integração entre previdência aberta e contexto socioeconômico

Os indicadores anuais da previdência aberta são combinados com os dados de população e rendimento médio mensal do IBGE por unidade da Federação e ano.

A contribuição anual é dividida pelo estoque de participantes no encerramento do ano e por 12 meses, produzindo uma aproximação da contribuição média mensal por registro de participante.

Esse indicador deve ser interpretado como uma média aproximada, pois:

- o estoque de participantes corresponde ao último mês do ano;
- as contribuições representam o fluxo acumulado dos 12 meses;
- um indivíduo pode possuir mais de um plano ou vínculo;
- a fonte não permite identificar participantes únicos.

A relação entre a contribuição média e o rendimento médio será utilizada como referência empírica para os cenários de acumulação, sem representar a taxa individual efetiva de cada participante.

In [0]:
# Recarrega as duas tabelas Gold diretamente do catálogo.
df_gold_aberta = spark.table(
    "workspace.gold.previdencia_aberta_uf_ano"
)

df_gold_contexto = spark.table(
    "workspace.gold.contexto_socioeconomico_uf_ano"
)

# Combina os indicadores previdenciários com renda e população
# utilizando UF e ano como chaves.
df_gold_aberta_contexto = (
    df_gold_aberta.alias("previdencia")
    .join(
        df_gold_contexto.alias("contexto"),
        on=[
            F.col("previdencia.uf") == F.col("contexto.sigla_uf"),
            F.col("previdencia.ano") == F.col("contexto.ano")
        ],
        how="left"
    )
    .select(
        F.col("previdencia.ano").alias("ano"),
        F.col("previdencia.uf").alias("uf"),
        F.col("contexto.uf").alias("nome_uf"),
        F.col("previdencia.regiao").alias("regiao"),
        F.col("previdencia.tipo_produto").alias("tipo_produto"),
        F.col("contexto.populacao_total").alias("populacao_total"),
        F.col("contexto.rendimento_medio_mensal_reais").alias(
            "rendimento_medio_mensal_reais"
        ),
        F.col("previdencia.contribuicao_anual").alias(
            "contribuicao_anual"
        ),
        F.col("previdencia.beneficio_pago_anual").alias(
            "beneficio_pago_anual"
        ),
        F.col("previdencia.resgate_pago_anual").alias(
            "resgate_pago_anual"
        ),
        F.col("previdencia.quantidade_resgates_anual").alias(
            "quantidade_resgates_anual"
        ),
        F.col("previdencia.participantes_fim_ano").alias(
            "participantes_fim_ano"
        ),
        F.col("previdencia.beneficiarios_fim_ano").alias(
            "beneficiarios_fim_ano"
        )
    )

    # Calcula uma aproximação da contribuição mensal média
    # por registro de participante.
    .withColumn(
        "contribuicao_media_mensal_participante",
        F.when(
            F.col("participantes_fim_ano") > 0,
            (
                F.col("contribuicao_anual")
                / F.col("participantes_fim_ano")
                / F.lit(12)
            ).cast("decimal(20,2)")
        )
    )

    # Relaciona a contribuição mensal média ao rendimento
    # médio mensal do respectivo estado.
    .withColumn(
        "percentual_contribuicao_sobre_rendimento",
        F.when(
            F.col("rendimento_medio_mensal_reais") > 0,
            (
                F.col("contribuicao_media_mensal_participante")
                / F.col("rendimento_medio_mensal_reais")
                * 100
            ).cast("decimal(12,4)")
        )
    )

    # Mede a relação anual entre resgates e contribuições.
    .withColumn(
        "percentual_resgates_sobre_contribuicoes",
        F.when(
            F.col("contribuicao_anual") > 0,
            (
                F.col("resgate_pago_anual")
                / F.col("contribuicao_anual")
                * 100
            ).cast("decimal(12,4)")
        )
    )

    # Calcula o fluxo líquido comum aos dois produtos,
    # considerando contribuições menos resgates.
    .withColumn(
        "contribuicoes_menos_resgates",
        (
            F.col("contribuicao_anual")
            - F.col("resgate_pago_anual")
        ).cast("decimal(20,2)")
    )

    # Para PREVTRAD, calcula também o resultado dos três
    # fluxos informados. No PGBL, permanece nulo porque
    # benefícios pagos não estão disponíveis nessa fonte.
    .withColumn(
        "resultado_fluxos_informados",
        F.when(
            F.col("beneficio_pago_anual").isNotNull(),
            (
                F.col("contribuicao_anual")
                - F.col("resgate_pago_anual")
                - F.col("beneficio_pago_anual")
            ).cast("decimal(20,2)")
        )
    )

    .withColumn("data_processamento", F.current_timestamp())
    .orderBy("ano", "uf", "tipo_produto")
)

display(df_gold_aberta_contexto)

ano,uf,nome_uf,regiao,tipo_produto,populacao_total,rendimento_medio_mensal_reais,contribuicao_anual,beneficio_pago_anual,resgate_pago_anual,quantidade_resgates_anual,participantes_fim_ano,beneficiarios_fim_ano,contribuicao_media_mensal_participante,percentual_contribuicao_sobre_rendimento,percentual_resgates_sobre_contribuicoes,contribuicoes_menos_resgates,resultado_fluxos_informados,data_processamento
2012,ES,Espírito Santo,Sudeste,PGBL,3711000,2820.00,96947396.16,null,48684991.86,5731,36177,null,223.32,7.9191,50.2179,48262404.30,null,2026-09-13T03:20:31.528Z
2012,ES,Espírito Santo,Sudeste,PREVTRAD,3711000,2820.00,44137529.45,12428095.72,11706045.58,1191,64023,869,57.45,2.0372,26.5218,32431483.87,20003388.15,2026-09-13T03:20:31.528Z
2012,MG,Minas Gerais,Sudeste,PGBL,20084000,2739.00,484803388.84,null,211647011.58,20729,106243,null,380.26,13.8832,43.6563,273156377.26,null,2026-09-13T03:20:31.528Z
2012,MG,Minas Gerais,Sudeste,PREVTRAD,20084000,2739.00,287767270.24,91861237.08,56461182.18,4120,404438,4677,59.29,2.1647,19.6204,231306088.06,139444850.98,2026-09-13T03:20:31.528Z
2012,RJ,Rio de Janeiro,Sudeste,PGBL,16719000,3332.00,961132114.63,null,447780125.00,35240,162744,null,492.15,14.7704,46.5888,513351989.63,null,2026-09-13T03:20:31.528Z
2012,RJ,Rio de Janeiro,Sudeste,PREVTRAD,16719000,3332.00,451733055.78,176869977.53,106221327.08,62685,533359,16330,70.58,2.1182,23.5142,345511728.70,168641751.17,2026-09-13T03:20:31.528Z
2012,SP,São Paulo,Sudeste,PGBL,43157000,3994.00,3924069664.57,null,2091783647.07,144362,939006,null,348.25,8.7193,53.3065,1832286017.50,null,2026-09-13T03:20:31.528Z
2012,SP,São Paulo,Sudeste,PREVTRAD,43157000,3994.00,1301963816.57,642888964.72,820786261.70,326554,1371059,23458,79.13,1.9812,63.0422,481177554.87,-161711409.85,2026-09-13T03:20:31.528Z
2013,ES,Espírito Santo,Sudeste,PGBL,3750000,3024.00,103710010.36,null,62624275.34,5739,41442,null,208.54,6.8962,60.3840,41085735.02,null,2026-09-13T03:20:31.528Z
2013,ES,Espírito Santo,Sudeste,PREVTRAD,3750000,3024.00,51196750.45,12135590.65,12072866.68,1085,74137,881,57.55,1.9031,23.5813,39123883.77,26988293.12,2026-09-13T03:20:31.528Z


### 6.1 Validação dos indicadores integrados

A integração é validada quanto à correspondência entre as unidades da Federação, anos e produtos, à presença das variáveis socioeconômicas e à unicidade das chaves.

Também são examinados os valores mínimos e máximos das aproximações de contribuição mensal e das relações entre contribuições, rendimento e resgates. Valores elevados não são excluídos automaticamente, pois podem refletir características dos dados agregados ou diferenças entre fluxo anual e estoque de participantes.

In [0]:
# Valida a estrutura, o cruzamento com o IBGE
# e o preenchimento dos principais indicadores.
df_validacao_aberta_contexto = (
    df_gold_aberta_contexto
    .agg(
        F.count("*").alias("registros"),
        F.countDistinct("uf").alias("ufs_distintas"),
        F.countDistinct("tipo_produto").alias(
            "produtos_distintos"
        ),
        F.min("ano").alias("primeiro_ano"),
        F.max("ano").alias("ultimo_ano"),

        # Valores socioeconômicos nulos indicariam
        # falha no cruzamento com a tabela de contexto.
        F.sum(
            F.when(F.col("nome_uf").isNull(), 1).otherwise(0)
        ).alias("ufs_sem_correspondencia"),
        F.sum(
            F.when(F.col("populacao_total").isNull(), 1).otherwise(0)
        ).alias("populacoes_nulas"),
        F.sum(
            F.when(
                F.col("rendimento_medio_mensal_reais").isNull(),
                1
            ).otherwise(0)
        ).alias("rendimentos_nulos"),

        # Confere os indicadores calculados.
        F.sum(
            F.when(
                F.col("contribuicao_media_mensal_participante").isNull(),
                1
            ).otherwise(0)
        ).alias("contribuicoes_medias_nulas"),
        F.sum(
            F.when(
                F.col("percentual_contribuicao_sobre_rendimento").isNull(),
                1
            ).otherwise(0)
        ).alias("percentuais_contribuicao_nulos"),
        F.sum(
            F.when(
                F.col("percentual_resgates_sobre_contribuicoes").isNull(),
                1
            ).otherwise(0)
        ).alias("percentuais_resgates_nulos")
    )
)

display(df_validacao_aberta_contexto)

registros,ufs_distintas,produtos_distintos,primeiro_ano,ultimo_ano,ufs_sem_correspondencia,populacoes_nulas,rendimentos_nulos,contribuicoes_medias_nulas,percentuais_contribuicao_nulos,percentuais_resgates_nulos
112,4,2,2012,2025,0,0,0,0,0,0


In [0]:
# Resume os principais indicadores por produto,
# permitindo identificar suas faixas de variação histórica.
df_resumo_indicadores_aberta = (
    df_gold_aberta_contexto
    .groupBy("tipo_produto")
    .agg(
        F.count("*").alias("registros"),

        F.min("contribuicao_media_mensal_participante").alias(
            "menor_contribuicao_media"
        ),
        F.max("contribuicao_media_mensal_participante").alias(
            "maior_contribuicao_media"
        ),

        F.min("percentual_contribuicao_sobre_rendimento").alias(
            "menor_percentual_contribuicao"
        ),
        F.max("percentual_contribuicao_sobre_rendimento").alias(
            "maior_percentual_contribuicao"
        ),

        F.min("percentual_resgates_sobre_contribuicoes").alias(
            "menor_percentual_resgates"
        ),
        F.max("percentual_resgates_sobre_contribuicoes").alias(
            "maior_percentual_resgates"
        ),

        F.sum(
            F.when(
                F.col("resultado_fluxos_informados") < 0,
                1
            ).otherwise(0)
        ).alias("anos_uf_com_resultado_negativo")
    )
    .orderBy("tipo_produto")
)

display(df_resumo_indicadores_aberta)

tipo_produto,registros,menor_contribuicao_media,maior_contribuicao_media,menor_percentual_contribuicao,maior_percentual_contribuicao,menor_percentual_resgates,maior_percentual_resgates,anos_uf_com_resultado_negativo
PGBL,56,166.71,907.59,4.2532,23.8463,43.6563,182.7821,0
PREVTRAD,56,3.55,107.72,0.0993,3.6529,14.9328,233.1603,29


In [0]:
# Verifica a unicidade por ano, UF e produto.
df_repeticoes_aberta_contexto = (
    df_gold_aberta_contexto
    .groupBy("ano", "uf", "tipo_produto")
    .count()
    .filter(F.col("count") > 1)
    .agg(
        F.count("*").alias("grupos_repetidos"),
        F.sum(F.col("count") - 1).alias("linhas_excedentes"),
        F.max("count").alias("maior_repeticao")
    )
)

display(df_repeticoes_aberta_contexto)

grupos_repetidos,linhas_excedentes,maior_repeticao
0,null,null


### 6.2 Gravação dos indicadores integrados

Após a validação da correspondência entre SUSEP e IBGE, os indicadores integrados são armazenados na camada Gold.

As métricas de contribuição média e percentual sobre o rendimento são aproximações construídas a partir de dados agregados. Elas serão utilizadas como referências empíricas para os cenários e não como taxas individuais observadas.

In [0]:
# Recupera os resultados das validações anteriores.
validacao_aberta_contexto = (
    df_validacao_aberta_contexto.first()
)

validacao_repeticoes_aberta_contexto = (
    df_repeticoes_aberta_contexto.first()
)

# Confirma a estrutura da tabela.
assert validacao_aberta_contexto["registros"] == 112, \
    "Quantidade inesperada de registros."

assert validacao_aberta_contexto["ufs_distintas"] == 4, \
    "Quantidade inesperada de UFs."

assert validacao_aberta_contexto["produtos_distintos"] == 2, \
    "Quantidade inesperada de produtos."

assert validacao_aberta_contexto["primeiro_ano"] == 2012, \
    "Primeiro ano inesperado."

assert validacao_aberta_contexto["ultimo_ano"] == 2025, \
    "Último ano inesperado."

# Confirma que o cruzamento com o IBGE foi completo.
assert validacao_aberta_contexto["ufs_sem_correspondencia"] == 0, \
    "Foram encontradas UFs sem correspondência."

assert validacao_aberta_contexto["populacoes_nulas"] == 0, \
    "Foram encontradas populações nulas."

assert validacao_aberta_contexto["rendimentos_nulos"] == 0, \
    "Foram encontrados rendimentos nulos."

# Confirma que os indicadores principais foram calculados.
assert validacao_aberta_contexto["contribuicoes_medias_nulas"] == 0, \
    "Foram encontradas contribuições médias nulas."

assert (
    validacao_aberta_contexto["percentuais_contribuicao_nulos"]
    == 0
), "Foram encontrados percentuais de contribuição nulos."

assert validacao_aberta_contexto["percentuais_resgates_nulos"] == 0, \
    "Foram encontrados percentuais de resgates nulos."

assert validacao_repeticoes_aberta_contexto["grupos_repetidos"] == 0, \
    "Foram encontradas duplicidades."

# Define o nome da tabela Gold.
tabela_gold_aberta_contexto = (
    "workspace.gold.indicadores_aberta_contexto_uf_ano"
)

# Grava a tabela em formato Delta após as validações.
(
    df_gold_aberta_contexto
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(tabela_gold_aberta_contexto)
)

print(
    "Validações aprovadas e tabela Gold gravada com sucesso:",
    tabela_gold_aberta_contexto
)

Validações aprovadas e tabela Gold gravada com sucesso: workspace.gold.indicadores_aberta_contexto_uf_ano


In [0]:
# Lê novamente a tabela diretamente da camada Gold.
df_aberta_contexto_gravado = spark.table(
    "workspace.gold.indicadores_aberta_contexto_uf_ano"
)

# Confere a tabela efetivamente armazenada.
df_validacao_aberta_contexto_gravado = (
    df_aberta_contexto_gravado
    .agg(
        F.count("*").alias("quantidade_registros"),
        F.countDistinct("uf").alias("ufs_distintas"),
        F.countDistinct("tipo_produto").alias(
            "produtos_distintos"
        ),
        F.min("ano").alias("primeiro_ano"),
        F.max("ano").alias("ultimo_ano")
    )
)

display(df_validacao_aberta_contexto_gravado)

quantidade_registros,ufs_distintas,produtos_distintos,primeiro_ano,ultimo_ano
112,4,2,2012,2025


## 7. Cenários de acumulação e reposição de renda

Os cenários estimam quanto uma contribuição mensal constante, mantida em termos reais, poderia acumular ao longo do tempo e qual percentual do rendimento atual poderia ser reposto durante a aposentadoria.

### 7.1 Referência empírica de contribuição

A referência de contribuição é obtida a partir do PGBL em 2025, dividindo-se a contribuição anual agregada pelo estoque de participantes no encerramento do ano e por 12 meses.

O PGBL foi selecionado porque possui informações de contribuição e participantes por unidade da Federação e está diretamente associado à acumulação previdenciária individual.

O PREVTRAD não será utilizado como referência principal da contribuição dos cenários porque representa uma carteira mais madura, com fluxos relevantes de benefícios e resgates. A baixa contribuição média observada nesse produto não representa necessariamente o esforço contributivo de um novo participante.

A previdência fechada também não será utilizada para estimar uma contribuição monetária média, pois as tabelas da PREVIC disponíveis neste projeto apresentam quantidades populacionais e movimentações de participantes, mas não valores de contribuição por indivíduo.

As projeções são ilustrativas, baseadas em dados agregados e premissas explícitas. Não representam recomendação financeira, garantia de rentabilidade ou previsão individual de benefício.

In [0]:
# Recarrega os indicadores integrados diretamente da Gold.
df_indicadores_aberta_contexto = spark.table(
    "workspace.gold.indicadores_aberta_contexto_uf_ano"
)

# Seleciona o PGBL e o último ano disponível como
# referência empírica para as simulações.
df_referencia_contribuicao_2025 = (
    df_indicadores_aberta_contexto
    .filter(
        (F.col("ano") == 2025)
        & (F.col("tipo_produto") == "PGBL")
    )
    .select(
        "ano",
        "uf",
        "nome_uf",
        "regiao",
        "rendimento_medio_mensal_reais",
        "participantes_fim_ano",
        "contribuicao_anual",
        "contribuicao_media_mensal_participante",
        "percentual_contribuicao_sobre_rendimento"
    )
    .orderBy("uf")
)

display(df_referencia_contribuicao_2025)

ano,uf,nome_uf,regiao,rendimento_medio_mensal_reais,participantes_fim_ano,contribuicao_anual,contribuicao_media_mensal_participante,percentual_contribuicao_sobre_rendimento
2025,ES,Espírito Santo,Sudeste,3497.00,61650,159978238.10,216.25,6.1839
2025,MG,Minas Gerais,Sudeste,3350.00,122415,1049408624.57,714.38,21.3248
2025,RJ,Rio de Janeiro,Sudeste,4177.00,152010,1613426892.26,884.50,21.1755
2025,SP,São Paulo,Sudeste,4190.00,2694083,8055211493.18,249.16,5.9465


In [0]:
# Resume a contribuição utilizada como referência
# nos quatro estados da Região Sudeste.
df_resumo_referencia_contribuicao = (
    df_referencia_contribuicao_2025
    .agg(
        F.count("*").alias("ufs"),
        F.min("contribuicao_media_mensal_participante").alias(
            "menor_contribuicao_mensal"
        ),
        F.max("contribuicao_media_mensal_participante").alias(
            "maior_contribuicao_mensal"
        ),
        F.avg("contribuicao_media_mensal_participante").alias(
            "media_contribuicao_mensal_ufs"
        ),
        F.min("percentual_contribuicao_sobre_rendimento").alias(
            "menor_percentual_renda"
        ),
        F.max("percentual_contribuicao_sobre_rendimento").alias(
            "maior_percentual_renda"
        )
    )
)

display(df_resumo_referencia_contribuicao)

ufs,menor_contribuicao_mensal,maior_contribuicao_mensal,media_contribuicao_mensal_ufs,menor_percentual_renda,maior_percentual_renda
4,216.25,884.50,516.072500,5.9465,21.3248


### 7.2 Premissas dos cenários

Os cenários utilizam a contribuição média mensal aproximada do PGBL em 2025 para cada unidade da Federação.

São considerados três horizontes de acumulação:

- 10 anos;
- 20 anos;
- 30 anos.

E três hipóteses de rentabilidade real anual durante a acumulação:

- conservadora: 2% ao ano;
- base: 4% ao ano;
- otimista: 6% ao ano.

Após a acumulação, o patrimônio é convertido em uma renda mensal durante 20 anos, considerando rentabilidade real de 2% ao ano na fase de recebimento.

As principais premissas são:

- contribuições realizadas ao final de cada mês;
- contribuição mensal constante em termos reais;
- ausência de saldo inicial;
- ausência de aportes extraordinários;
- ausência de resgates durante a acumulação;
- ausência de taxas administrativas, carregamento e tributação;
- valores expressos em reais de 2025;
- renda mensal calculada como anuidade financeira, sem garantia vitalícia.

Essas premissas simplificam a realidade e têm finalidade exclusivamente acadêmica e comparativa.

In [0]:
# Cria os três cenários de rentabilidade real anual
# utilizados durante o período de acumulação.
df_cenarios_rentabilidade = spark.createDataFrame(
    [
        ("CONSERVADOR", 0.02),
        ("BASE", 0.04),
        ("OTIMISTA", 0.06)
    ],
    ["cenario", "taxa_retorno_real_acumulacao_anual"]
)

# Cria os horizontes de contribuição analisados.
df_horizontes = spark.createDataFrame(
    [
        (10,),
        (20,),
        (30,)
    ],
    ["horizonte_anos"]
)

display(df_cenarios_rentabilidade)
display(df_horizontes)

cenario,taxa_retorno_real_acumulacao_anual
CONSERVADOR,0.02
BASE,0.04
OTIMISTA,0.06


horizonte_anos
10
20
30


In [0]:
from pyspark.sql import functions as F

df_indicadores_aberta_contexto = spark.table(
    "workspace.gold.indicadores_aberta_contexto_uf_ano"
)

df_referencia_contribuicao_2025 = (
    df_indicadores_aberta_contexto
    .filter(
        (F.col("ano") == 2025)
        & (F.col("tipo_produto") == "PGBL")
    )
    .select(
        "ano",
        "uf",
        "nome_uf",
        "regiao",
        "rendimento_medio_mensal_reais",
        "participantes_fim_ano",
        "contribuicao_anual",
        "contribuicao_media_mensal_participante",
        "percentual_contribuicao_sobre_rendimento"
    )
)


# Define as premissas comuns à fase de recebimento.
taxa_retorno_real_beneficio_anual = 0.02
duracao_beneficio_anos = 20

# Combina as quatro UFs com três cenários
# e três horizontes de acumulação.
df_gold_cenarios_reposicao = (
    df_referencia_contribuicao_2025
    .crossJoin(df_cenarios_rentabilidade)
    .crossJoin(df_horizontes)

    # Converte a taxa anual efetiva de acumulação
    # para sua taxa mensal equivalente.
    .withColumn(
        "taxa_retorno_real_acumulacao_mensal",
        F.pow(
            F.lit(1.0)
            + F.col("taxa_retorno_real_acumulacao_anual"),
            F.lit(1.0 / 12.0)
        ) - F.lit(1.0)
    )

    # Calcula a quantidade de contribuições mensais.
    .withColumn(
        "quantidade_meses_acumulacao",
        F.col("horizonte_anos") * F.lit(12)
    )

    # Calcula o total nominal de contribuições em reais constantes,
    # sem considerar os rendimentos acumulados.
    .withColumn(
        "total_contribuido",
        (
            F.col("contribuicao_media_mensal_participante")
            * F.col("quantidade_meses_acumulacao")
        ).cast("decimal(20,2)")
    )

    # Calcula o valor futuro de uma série de contribuições
    # realizadas ao final de cada mês.
    .withColumn(
        "patrimonio_acumulado",
        (
            F.col("contribuicao_media_mensal_participante")
            * (
                (
                    F.pow(
                        F.lit(1.0)
                        + F.col(
                            "taxa_retorno_real_acumulacao_mensal"
                        ),
                        F.col("quantidade_meses_acumulacao")
                    )
                    - F.lit(1.0)
                )
                / F.col(
                    "taxa_retorno_real_acumulacao_mensal"
                )
            )
        ).cast("decimal(20,2)")
    )

    # Calcula quanto do patrimônio decorre do retorno real
    # acima do total das contribuições realizadas.
    .withColumn(
        "rendimento_real_acumulado",
        (
            F.col("patrimonio_acumulado")
            - F.col("total_contribuido")
        ).cast("decimal(20,2)")
    )

    # Registra as premissas da fase de recebimento.
    .withColumn(
        "taxa_retorno_real_beneficio_anual",
        F.lit(taxa_retorno_real_beneficio_anual)
    )
    .withColumn(
        "duracao_beneficio_anos",
        F.lit(duracao_beneficio_anos)
    )

    # Converte a taxa anual da fase de benefício
    # para a taxa mensal equivalente.
    .withColumn(
        "taxa_retorno_real_beneficio_mensal",
        F.pow(
            F.lit(1.0)
            + F.col("taxa_retorno_real_beneficio_anual"),
            F.lit(1.0 / 12.0)
        ) - F.lit(1.0)
    )

    # Calcula a quantidade de pagamentos mensais
    # prevista para a aposentadoria.
    .withColumn(
        "quantidade_meses_beneficio",
        F.col("duracao_beneficio_anos") * F.lit(12)
    )

    # Converte o patrimônio acumulado em uma renda mensal
    # por meio da fórmula de uma anuidade financeira.
    .withColumn(
        "renda_mensal_projetada",
        (
            F.col("patrimonio_acumulado")
            * F.col("taxa_retorno_real_beneficio_mensal")
            / (
                F.lit(1.0)
                - F.pow(
                    F.lit(1.0)
                    + F.col("taxa_retorno_real_beneficio_mensal"),
                    -F.col("quantidade_meses_beneficio")
                )
            )
        ).cast("decimal(20,2)")
    )

    # Calcula o percentual do rendimento médio atual
    # que seria reposto pela renda previdenciária projetada.
    .withColumn(
        "taxa_reposicao_renda_percentual",
        (
            F.col("renda_mensal_projetada")
            / F.col("rendimento_medio_mensal_reais")
            * F.lit(100)
        ).cast("decimal(12,4)")
    )

    # Acrescenta informações metodológicas e de rastreabilidade.
    .withColumn(
        "metodo_calculo",
        F.lit("Anuidade mensal com contribuições postecipadas")
    )
    .withColumn("moeda_referencia", F.lit("Reais de 2025"))
    .withColumn("data_processamento", F.current_timestamp())

    .orderBy("uf", "horizonte_anos", "cenario")
)

display(df_gold_cenarios_reposicao)

ano,uf,nome_uf,regiao,rendimento_medio_mensal_reais,participantes_fim_ano,contribuicao_anual,contribuicao_media_mensal_participante,percentual_contribuicao_sobre_rendimento,cenario,taxa_retorno_real_acumulacao_anual,horizonte_anos,taxa_retorno_real_acumulacao_mensal,quantidade_meses_acumulacao,total_contribuido,patrimonio_acumulado,rendimento_real_acumulado,taxa_retorno_real_beneficio_anual,duracao_beneficio_anos,taxa_retorno_real_beneficio_mensal,quantidade_meses_beneficio,renda_mensal_projetada,taxa_reposicao_renda_percentual,metodo_calculo,moeda_referencia,data_processamento
2025,ES,Espírito Santo,Sudeste,3497.00,61650,159978238.10,216.25,6.1839,BASE,0.04,10,0.0032737397821989145,120,25950.00,31722.99,5772.99,0.02,20,0.0016515813019202241,240,160.21,4.5814,Anuidade mensal com contribuições postecipadas,Reais de 2025,2026-09-13T21:42:53.356Z
2025,ES,Espírito Santo,Sudeste,3497.00,61650,159978238.10,216.25,6.1839,CONSERVADOR,0.02,10,0.0016515813019202241,120,25950.00,28674.06,2724.06,0.02,20,0.0016515813019202241,240,144.81,4.1410,Anuidade mensal com contribuições postecipadas,Reais de 2025,2026-09-13T21:42:53.356Z
2025,ES,Espírito Santo,Sudeste,3497.00,61650,159978238.10,216.25,6.1839,OTIMISTA,0.06,10,0.004867550565343048,120,25950.00,35134.88,9184.88,0.02,20,0.0016515813019202241,240,177.44,5.0741,Anuidade mensal com contribuições postecipadas,Reais de 2025,2026-09-13T21:42:53.356Z
2025,ES,Espírito Santo,Sudeste,3497.00,61650,159978238.10,216.25,6.1839,BASE,0.04,20,0.0032737397821989145,240,51900.00,78680.77,26780.77,0.02,20,0.0016515813019202241,240,397.36,11.3629,Anuidade mensal com contribuições postecipadas,Reais de 2025,2026-09-13T21:42:53.356Z
2025,ES,Espírito Santo,Sudeste,3497.00,61650,159978238.10,216.25,6.1839,CONSERVADOR,0.02,20,0.0016515813019202241,240,51900.00,63627.58,11727.58,0.02,20,0.0016515813019202241,240,321.34,9.1890,Anuidade mensal com contribuições postecipadas,Reais de 2025,2026-09-13T21:42:53.356Z
2025,ES,Espírito Santo,Sudeste,3497.00,61650,159978238.10,216.25,6.1839,OTIMISTA,0.06,20,0.004867550565343048,240,51900.00,98056.10,46156.10,0.02,20,0.0016515813019202241,240,495.21,14.1610,Anuidade mensal com contribuições postecipadas,Reais de 2025,2026-09-13T21:42:53.356Z
2025,ES,Espírito Santo,Sudeste,3497.00,61650,159978238.10,216.25,6.1839,BASE,0.04,30,0.0032737397821989145,360,77850.00,148189.76,70339.76,0.02,20,0.0016515813019202241,240,748.40,21.4012,Anuidade mensal com contribuições postecipadas,Reais de 2025,2026-09-13T21:42:53.356Z
2025,ES,Espírito Santo,Sudeste,3497.00,61650,159978238.10,216.25,6.1839,CONSERVADOR,0.02,30,0.0016515813019202241,360,77850.00,106235.73,28385.73,0.02,20,0.0016515813019202241,240,536.52,15.3423,Anuidade mensal com contribuições postecipadas,Reais de 2025,2026-09-13T21:42:53.356Z
2025,ES,Espírito Santo,Sudeste,3497.00,61650,159978238.10,216.25,6.1839,OTIMISTA,0.06,30,0.004867550565343048,360,77850.00,210738.43,132888.43,0.02,20,0.0016515813019202241,240,1064.28,30.4341,Anuidade mensal com contribuições postecipadas,Reais de 2025,2026-09-13T21:42:53.356Z
2025,MG,Minas Gerais,Sudeste,3350.00,122415,1049408624.57,714.38,21.3248,BASE,0.04,10,0.0032737397821989145,120,85725.60,104796.63,19071.03,0.02,20,0.0016515813019202241,240,529.25,15.7985,Anuidade mensal com contribuições postecipadas,Reais de 2025,2026-09-13T21:42:53.356Z


### 7.3 Validação dos cenários de reposição

Os cenários são verificados quanto à quantidade de combinações, presença de valores, unicidade e coerência financeira.

São esperadas 36 combinações, formadas por quatro unidades da Federação, três horizontes e três cenários.

Também são avaliadas as seguintes condições:

- patrimônio acumulado não inferior ao total contribuído, considerando taxas reais positivas;
- renda mensal e taxa de reposição positivas;
- crescimento do patrimônio com o aumento do horizonte;
- crescimento do patrimônio com o aumento da rentabilidade;
- ausência de combinações repetidas.

In [0]:
# Valida a estrutura e os principais resultados financeiros.
df_validacao_cenarios_reposicao = (
    df_gold_cenarios_reposicao
    .agg(
        F.count("*").alias("registros"),
        F.countDistinct("uf").alias("ufs_distintas"),
        F.countDistinct("cenario").alias("cenarios_distintos"),
        F.countDistinct("horizonte_anos").alias(
            "horizontes_distintos"
        ),

        # Confere a ausência de valores nulos.
        F.sum(
            F.when(F.col("patrimonio_acumulado").isNull(), 1).otherwise(0)
        ).alias("patrimonios_nulos"),
        F.sum(
            F.when(F.col("renda_mensal_projetada").isNull(), 1).otherwise(0)
        ).alias("rendas_nulas"),
        F.sum(
            F.when(
                F.col("taxa_reposicao_renda_percentual").isNull(),
                1
            ).otherwise(0)
        ).alias("taxas_reposicao_nulas"),

        # Com taxas reais positivas, o patrimônio não deve
        # ficar abaixo do total contribuído.
        F.sum(
            F.when(
                F.col("patrimonio_acumulado")
                < F.col("total_contribuido"),
                1
            ).otherwise(0)
        ).alias("patrimonios_inferiores_contribuicoes"),

        # Confere valores financeiros não positivos.
        F.sum(
            F.when(
                F.col("renda_mensal_projetada") <= 0,
                1
            ).otherwise(0)
        ).alias("rendas_nao_positivas"),
        F.sum(
            F.when(
                F.col("taxa_reposicao_renda_percentual") <= 0,
                1
            ).otherwise(0)
        ).alias("taxas_reposicao_nao_positivas")
    )
)

display(df_validacao_cenarios_reposicao)

registros,ufs_distintas,cenarios_distintos,horizontes_distintos,patrimonios_nulos,rendas_nulas,taxas_reposicao_nulas,patrimonios_inferiores_contribuicoes,rendas_nao_positivas,taxas_reposicao_nao_positivas
36,4,3,3,0,0,0,0,0,0


In [0]:
# Importa a função necessária para realizar
# comparações entre linhas ordenadas.
from pyspark.sql.window import Window

# Define uma janela para verificar se o patrimônio cresce
# quando o horizonte aumenta dentro do mesmo cenário.
janela_crescimento_horizonte = (
    Window
    .partitionBy("uf", "cenario")
    .orderBy("horizonte_anos")
)

# Define uma janela para verificar se o patrimônio cresce
# quando a taxa de retorno aumenta no mesmo horizonte.
janela_crescimento_retorno = (
    Window
    .partitionBy("uf", "horizonte_anos")
    .orderBy("taxa_retorno_real_acumulacao_anual")
)

df_coerencia_cenarios = (
    df_gold_cenarios_reposicao

    .withColumn(
        "patrimonio_horizonte_anterior",
        F.lag("patrimonio_acumulado").over(
            janela_crescimento_horizonte
        )
    )

    .withColumn(
        "patrimonio_retorno_anterior",
        F.lag("patrimonio_acumulado").over(
            janela_crescimento_retorno
        )
    )
)

# Conta eventuais violações das relações esperadas.
df_validacao_coerencia_cenarios = (
    df_coerencia_cenarios
    .agg(
        F.sum(
            F.when(
                F.col("patrimonio_horizonte_anterior").isNotNull()
                & (
                    F.col("patrimonio_acumulado")
                    <= F.col("patrimonio_horizonte_anterior")
                ),
                1
            ).otherwise(0)
        ).alias("violacoes_crescimento_horizonte"),

        F.sum(
            F.when(
                F.col("patrimonio_retorno_anterior").isNotNull()
                & (
                    F.col("patrimonio_acumulado")
                    <= F.col("patrimonio_retorno_anterior")
                ),
                1
            ).otherwise(0)
        ).alias("violacoes_crescimento_retorno")
    )
)

display(df_validacao_coerencia_cenarios)

violacoes_crescimento_horizonte,violacoes_crescimento_retorno
0,0


In [0]:
# Verifica a unicidade de UF, cenário e horizonte.
df_repeticoes_cenarios = (
    df_gold_cenarios_reposicao
    .groupBy("uf", "cenario", "horizonte_anos")
    .count()
    .filter(F.col("count") > 1)
    .agg(
        F.count("*").alias("grupos_repetidos"),
        F.sum(F.col("count") - 1).alias("linhas_excedentes"),
        F.max("count").alias("maior_repeticao")
    )
)

display(df_repeticoes_cenarios)

grupos_repetidos,linhas_excedentes,maior_repeticao
0,null,null


### 7.4 Gravação dos cenários de reposição de renda

Após a validação estrutural e financeira, os 36 cenários são armazenados na camada Gold.

A tabela mantém as referências de contribuição e rendimento, as premissas utilizadas, o total contribuído, o patrimônio acumulado, a renda mensal projetada e a taxa estimada de reposição da renda.

In [0]:
# Recupera os resultados das validações.
validacao_cenarios = df_validacao_cenarios_reposicao.first()
validacao_coerencia = df_validacao_coerencia_cenarios.first()
validacao_repeticoes = df_repeticoes_cenarios.first()

# Confirma a estrutura esperada.
assert validacao_cenarios["registros"] == 36, \
    "Quantidade inesperada de cenários."

assert validacao_cenarios["ufs_distintas"] == 4, \
    "Quantidade inesperada de UFs."

assert validacao_cenarios["cenarios_distintos"] == 3, \
    "Quantidade inesperada de cenários de rentabilidade."

assert validacao_cenarios["horizontes_distintos"] == 3, \
    "Quantidade inesperada de horizontes."

# Confirma a integridade dos resultados.
assert validacao_cenarios["patrimonios_nulos"] == 0, \
    "Foram encontrados patrimônios nulos."

assert validacao_cenarios["rendas_nulas"] == 0, \
    "Foram encontradas rendas projetadas nulas."

assert validacao_cenarios["taxas_reposicao_nulas"] == 0, \
    "Foram encontradas taxas de reposição nulas."

assert (
    validacao_cenarios["patrimonios_inferiores_contribuicoes"]
    == 0
), "Existe patrimônio inferior ao total contribuído."

assert validacao_cenarios["rendas_nao_positivas"] == 0, \
    "Foram encontradas rendas não positivas."

assert validacao_cenarios["taxas_reposicao_nao_positivas"] == 0, \
    "Foram encontradas taxas de reposição não positivas."

# Confirma a coerência entre prazo, retorno e patrimônio.
assert (
    validacao_coerencia["violacoes_crescimento_horizonte"]
    == 0
), "O patrimônio não cresceu com o horizonte."

assert (
    validacao_coerencia["violacoes_crescimento_retorno"]
    == 0
), "O patrimônio não cresceu com a rentabilidade."

assert validacao_repeticoes["grupos_repetidos"] == 0, \
    "Foram encontrados cenários repetidos."

# Define o nome da tabela de destino.
tabela_gold_cenarios_reposicao = (
    "workspace.gold.cenarios_reposicao_renda_2025"
)

# Grava os cenários somente após todas as validações.
(
    df_gold_cenarios_reposicao
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(tabela_gold_cenarios_reposicao)
)

print(
    "Validações aprovadas e tabela Gold gravada com sucesso:",
    tabela_gold_cenarios_reposicao
)

Validações aprovadas e tabela Gold gravada com sucesso: workspace.gold.cenarios_reposicao_renda_2025


In [0]:
# Lê os cenários diretamente da camada Gold.
df_cenarios_reposicao_gravado = spark.table(
    "workspace.gold.cenarios_reposicao_renda_2025"
)

# Confere a tabela efetivamente armazenada.
df_validacao_cenarios_gravado = (
    df_cenarios_reposicao_gravado
    .agg(
        F.count("*").alias("quantidade_registros"),
        F.countDistinct("uf").alias("ufs_distintas"),
        F.countDistinct("cenario").alias("cenarios_distintos"),
        F.countDistinct("horizonte_anos").alias(
            "horizontes_distintos"
        ),
        F.min("taxa_reposicao_renda_percentual").alias(
            "menor_taxa_reposicao"
        ),
        F.max("taxa_reposicao_renda_percentual").alias(
            "maior_taxa_reposicao"
        )
    )
)

display(df_validacao_cenarios_gravado)

quantidade_registros,ufs_distintas,cenarios_distintos,horizontes_distintos,menor_taxa_reposicao,maior_taxa_reposicao
36,4,3,3,3.9821,104.9510


## 8. Resumo comparativo dos cenários

Os resultados são reorganizados para apresentar, em uma única linha, os cenários conservador, base e otimista de cada unidade da Federação e horizonte.

Essa estrutura facilita a elaboração de gráficos e a comparação do patrimônio, da renda mensal projetada e da taxa de reposição.

Taxas de reposição superiores a 100% significam que, sob as premissas adotadas, a renda previdenciária projetada supera o rendimento médio utilizado como referência. Esse resultado não representa garantia de benefício futuro.

In [0]:
from pyspark.sql import functions as F

# Recarrega os cenários diretamente da camada Gold.
df_cenarios_reposicao = spark.table(
    "workspace.gold.cenarios_reposicao_renda_2025"
)

# Organiza os três cenários em colunas para facilitar
# comparações, análises e visualizações.
df_gold_resumo_cenarios = (
    df_cenarios_reposicao

    .groupBy(
        "ano",
        "uf",
        "nome_uf",
        "regiao",
        "horizonte_anos",
        "rendimento_medio_mensal_reais",
        "contribuicao_media_mensal_participante",
        "percentual_contribuicao_sobre_rendimento"
    )

    .agg(
        # Patrimônio acumulado por cenário.
        F.max(
            F.when(
                F.col("cenario") == "CONSERVADOR",
                F.col("patrimonio_acumulado")
            )
        ).alias("patrimonio_conservador"),

        F.max(
            F.when(
                F.col("cenario") == "BASE",
                F.col("patrimonio_acumulado")
            )
        ).alias("patrimonio_base"),

        F.max(
            F.when(
                F.col("cenario") == "OTIMISTA",
                F.col("patrimonio_acumulado")
            )
        ).alias("patrimonio_otimista"),

        # Renda mensal projetada por cenário.
        F.max(
            F.when(
                F.col("cenario") == "CONSERVADOR",
                F.col("renda_mensal_projetada")
            )
        ).alias("renda_mensal_conservadora"),

        F.max(
            F.when(
                F.col("cenario") == "BASE",
                F.col("renda_mensal_projetada")
            )
        ).alias("renda_mensal_base"),

        F.max(
            F.when(
                F.col("cenario") == "OTIMISTA",
                F.col("renda_mensal_projetada")
            )
        ).alias("renda_mensal_otimista"),

        # Taxa de reposição da renda por cenário.
        F.max(
            F.when(
                F.col("cenario") == "CONSERVADOR",
                F.col("taxa_reposicao_renda_percentual")
            )
        ).alias("taxa_reposicao_conservadora"),

        F.max(
            F.when(
                F.col("cenario") == "BASE",
                F.col("taxa_reposicao_renda_percentual")
            )
        ).alias("taxa_reposicao_base"),

        F.max(
            F.when(
                F.col("cenario") == "OTIMISTA",
                F.col("taxa_reposicao_renda_percentual")
            )
        ).alias("taxa_reposicao_otimista")
    )

    # Renomeia os valores de referência para explicitar
    # que permanecem constantes em reais de 2025.
    .withColumnRenamed(
        "rendimento_medio_mensal_reais",
        "rendimento_referencia_reais_2025"
    )

    .withColumnRenamed(
        "contribuicao_media_mensal_participante",
        "contribuicao_mensal_constante_reais_2025"
    )

    .withColumn(
        "data_processamento",
        F.current_timestamp()
    )

    .orderBy(
        "uf",
        "horizonte_anos"
    )
)

display(df_gold_resumo_cenarios)

ano,uf,nome_uf,regiao,horizonte_anos,rendimento_referencia_reais_2025,contribuicao_mensal_constante_reais_2025,percentual_contribuicao_sobre_rendimento,patrimonio_conservador,patrimonio_base,patrimonio_otimista,renda_mensal_conservadora,renda_mensal_base,renda_mensal_otimista,taxa_reposicao_conservadora,taxa_reposicao_base,taxa_reposicao_otimista,data_processamento
2025,ES,Espírito Santo,Sudeste,10,3497.00,216.25,6.1839,28674.06,31722.99,35134.88,144.81,160.21,177.44,4.1410,4.5814,5.0741,2026-09-13T22:26:15.789Z
2025,ES,Espírito Santo,Sudeste,20,3497.00,216.25,6.1839,63627.58,78680.77,98056.10,321.34,397.36,495.21,9.1890,11.3629,14.1610,2026-09-13T22:26:15.789Z
2025,ES,Espírito Santo,Sudeste,30,3497.00,216.25,6.1839,106235.73,148189.76,210738.43,536.52,748.40,1064.28,15.3423,21.4012,30.4341,2026-09-13T22:26:15.789Z
2025,MG,Minas Gerais,Sudeste,10,3350.00,714.38,21.3248,94724.51,104796.63,116067.78,478.38,529.25,586.17,14.2800,15.7985,17.4976,2026-09-13T22:26:15.789Z
2025,MG,Minas Gerais,Sudeste,20,3350.00,714.38,21.3248,210193.16,259921.25,323927.49,1061.53,1312.67,1635.92,31.6875,39.1842,48.8334,2026-09-13T22:26:15.789Z
2025,MG,Minas Gerais,Sudeste,30,3350.00,714.38,21.3248,350948.81,489543.59,696172.58,1772.38,2472.32,3515.86,52.9069,73.8006,104.9510,2026-09-13T22:26:15.789Z
2025,RJ,Rio de Janeiro,Sudeste,10,4177.00,884.50,21.1755,117281.88,129752.55,143707.76,592.30,655.28,725.76,14.1800,15.6878,17.3751,2026-09-13T22:26:15.789Z
2025,RJ,Rio de Janeiro,Sudeste,20,4177.00,884.50,21.1755,260247.84,321818.01,401066.47,1314.32,1625.27,2025.49,31.4656,38.9100,48.4915,2026-09-13T22:26:15.789Z
2025,RJ,Rio de Janeiro,Sudeste,30,4177.00,884.50,21.1755,434522.55,606121.81,861956.72,2194.45,3061.08,4353.11,52.5365,73.2842,104.2162,2026-09-13T22:26:15.789Z
2025,SP,São Paulo,Sudeste,10,4190.00,249.16,5.9465,33037.82,36550.76,40481.88,166.85,184.59,204.44,3.9821,4.4055,4.8792,2026-09-13T22:26:15.789Z


In [0]:
# Valida a estrutura e a coerência do resumo comparativo.
df_validacao_resumo_cenarios = (
    df_gold_resumo_cenarios
    .agg(
        F.count("*").alias("registros"),
        F.countDistinct("uf").alias("ufs_distintas"),
        F.countDistinct("horizonte_anos").alias(
            "horizontes_distintos"
        ),

        # Confere se todos os cenários foram preenchidos.
        F.sum(
            F.when(
                F.col("taxa_reposicao_conservadora").isNull()
                | F.col("taxa_reposicao_base").isNull()
                | F.col("taxa_reposicao_otimista").isNull(),
                1
            ).otherwise(0)
        ).alias("registros_com_cenario_nulo"),

        # Verifica se a ordenação dos resultados é coerente:
        # conservador menor que base e base menor que otimista.
        F.sum(
            F.when(
                ~(
                    (
                        F.col("taxa_reposicao_conservadora")
                        < F.col("taxa_reposicao_base")
                    )
                    & (
                        F.col("taxa_reposicao_base")
                        < F.col("taxa_reposicao_otimista")
                    )
                ),
                1
            ).otherwise(0)
        ).alias("violacoes_ordem_cenarios")
    )
)

display(df_validacao_resumo_cenarios)

registros,ufs_distintas,horizontes_distintos,registros_com_cenario_nulo,violacoes_ordem_cenarios
12,4,3,0,0


In [0]:
# Recupera o resultado da validação.
validacao_resumo = df_validacao_resumo_cenarios.first()

# Interrompe a gravação caso alguma validação não seja atendida.
assert validacao_resumo["registros"] == 12, \
    "Quantidade inesperada de registros."

assert validacao_resumo["ufs_distintas"] == 4, \
    "Quantidade inesperada de UFs."

assert validacao_resumo["horizontes_distintos"] == 3, \
    "Quantidade inesperada de horizontes."

assert validacao_resumo["registros_com_cenario_nulo"] == 0, \
    "Existem registros com cenário nulo."

assert validacao_resumo["violacoes_ordem_cenarios"] == 0, \
    "Existem violações na ordem dos cenários."

# Define o nome da tabela definitiva.
tabela_gold_resumo_cenarios = (
    "workspace.gold.resumo_cenarios_reposicao_uf"
)

# Grava o resumo comparativo na camada Gold.
(
    df_gold_resumo_cenarios
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(tabela_gold_resumo_cenarios)
)

print(
    f"Tabela gravada com sucesso: "
    f"{tabela_gold_resumo_cenarios}"
)

Tabela gravada com sucesso: workspace.gold.resumo_cenarios_reposicao_uf


In [0]:
# Lê novamente o resumo diretamente da camada Gold.
df_resumo_cenarios_gravado = spark.table(
    "workspace.gold.resumo_cenarios_reposicao_uf"
)

# Confere a tabela efetivamente armazenada.
df_validacao_resumo_gravado = (
    df_resumo_cenarios_gravado
    .agg(
        F.count("*").alias("quantidade_registros"),
        F.countDistinct("uf").alias("ufs_distintas"),
        F.countDistinct("horizonte_anos").alias(
            "horizontes_distintos"
        ),
        F.min("taxa_reposicao_conservadora").alias(
            "menor_taxa_reposicao"
        ),
        F.max("taxa_reposicao_otimista").alias(
            "maior_taxa_reposicao"
        )
    )
)

display(df_validacao_resumo_gravado)

quantidade_registros,ufs_distintas,horizontes_distintos,menor_taxa_reposicao,maior_taxa_reposicao
12,4,3,3.9821,104.9510


In [0]:
# Lista todas as tabelas construídas na camada Gold.
df_inventario_gold = spark.sql(
    """
    SHOW TABLES IN workspace.gold
    """
)

display(
    df_inventario_gold
    .select("tableName")
    .orderBy("tableName")
)

tableName
cenarios_reposicao_renda_2025
contexto_socioeconomico_uf_ano
indicadores_aberta_contexto_uf_ano
movimentacao_plano_entidade_fechada_2025
perfil_previdencia_fechada_2025
previdencia_aberta_uf_ano
resumo_cenarios_reposicao_uf


## Conclusão da modelagem da camada Gold

A camada Gold consolidou os dados tratados nas camadas anteriores em tabelas analíticas voltadas às perguntas de negócio do projeto. Foram produzidas sete tabelas, abrangendo o contexto socioeconômico da Região Sudeste, os indicadores da previdência complementar aberta, o perfil e a movimentação da previdência complementar fechada e as projeções de reposição de renda.

### Previdência complementar aberta

Os dados da SUSEP foram combinados com informações de população e rendimento do IBGE para os estados do Espírito Santo, Minas Gerais, Rio de Janeiro e São Paulo. Essa integração permitiu calcular indicadores anuais de contribuição, resgates, participantes e contribuição média mensal por participante.

Para as projeções, foram utilizados como referência os valores observados em 2025 para o produto PGBL. O rendimento mensal e a contribuição mensal permanecem constantes entre os horizontes porque estão expressos em reais constantes de 2025. Portanto, o modelo não considera crescimento real do salário acima da inflação.

### Cenários de reposição de renda

Foram simulados três cenários de rentabilidade real anual:

- conservador: 2%;
- base: 4%;
- otimista: 6%.

As projeções consideram horizontes de acumulação de 10, 20 e 30 anos, contribuições mensais postecipadas e duração do benefício de 20 anos. Durante o período de recebimento foi adotada rentabilidade real anual de 2%.

Os resultados demonstram que o tempo de acumulação e a rentabilidade exercem impacto relevante sobre o patrimônio acumulado e sobre a renda mensal projetada. A taxa de reposição calculada variou entre aproximadamente 3,98% e 104,95%, considerando todas as combinações de UF, horizonte e cenário.

### Previdência complementar fechada

A previdência fechada foi analisada em âmbito nacional, pois os dados públicos utilizados não apresentam identificação geográfica por Unidade da Federação. Foram produzidos indicadores de perfil populacional e de movimentação por plano e entidade, mantendo registros com cobertura incompleta ou diferenças de conciliação identificados por flags de qualidade.

### Limitações e interpretação

As projeções possuem finalidade acadêmica e ilustrativa e não representam garantia de rentabilidade ou recomendação financeira. O modelo não considera saldo inicial, crescimento real dos salários, contribuições extraordinárias, resgates durante a acumulação, taxas administrativas, tributação ou alterações futuras no valor da contribuição.

Dessa forma, os resultados devem ser interpretados como cenários comparativos destinados a demonstrar como diferentes horizontes e taxas de retorno podem afetar a capacidade da previdência complementar de repor parte da renda utilizada como referência.